# S14 — Cas pratique Olist

## Analyse de la performance des vendeurs et des produits

### Mission

L'équipe Analytics d'Olist demande une analyse complète de la performance des
vendeurs et des produits de la marketplace.

L'objectif est de partir des données brutes, d'explorer chaque table séparément,
de contrôler leur qualité et leur cardinalité, puis de construire un jeu de
données enrichi permettant de répondre à des questions commerciales.

### Questions métier

- Quelles catégories génèrent le plus de chiffre d'affaires ?
- Quels vendeurs génèrent le plus de chiffre d'affaires ?
- Où sont situés les vendeurs les plus importants ?
- Quelle satisfaction client est associée aux ventes ?
- Les ventes sont-elles plutôt locales ou nationales ?
- Comment le chiffre d'affaires évolue-t-il dans le temps ?
- Quelles informations peuvent être transformées en recommandations commerciales ?

### Outils Python principaux

Ce cas pratique met particulièrement en œuvre :

- `merge()`
- `groupby()`
- `pivot_table()`
- `apply()`
- `map()`

> **Important : aucune visualisation n'est demandée cette semaine.**
> Le livrable repose sur l'exploration, les tableaux de synthèse,
> l'interprétation et la recommandation.

# Organisation du notebook

Le travail suit l'ordre demandé dans le cas pratique :

1. Chargement des données
2. Partie 0 — Nouvelles notions
3. Partie EDA — Explorer avant de croiser
4. Partie A — `merge`
5. Partie B — `groupby`
6. Partie C — `pivot_table`
7. Partie D — `apply` et `map`
8. Exercice 17 — Synthèse commerciale
9. Contrôles finaux

Le principe fondamental est :

> **Explorer et comprendre chaque table avant de la fusionner avec une autre.**

# PRÉREQUIS TECHNIQUE — Chargement des données

Le dataset utilisé est :

`olistbr/brazilian-ecommerce`

Il contient sept tables principales :

| Table | Grain |
|---|---|
| `orders` | une ligne = une commande |
| `items` | une ligne = un produit vendu dans une commande |
| `products` | une ligne = un produit du catalogue |
| `cat_trans` | une ligne = une catégorie traduite |
| `sellers` | une ligne = un vendeur |
| `customers` | une ligne = un client |
| `reviews` | une ligne = un avis |

Deux méthodes de chargement sont présentées :

1. chargement manuel des CSV ;
2. chargement avec `kagglehub`.

La méthode CSV reste volontairement commentée, conformément au sujet.

In [1]:
# ============================================================
# MÉTHODE 1 — CHARGEMENT MANUEL DES CSV
# ============================================================
#
# Cette méthode correspond à la méthode utilisée précédemment
# avec des fichiers CSV téléchargés localement.
#
# Décommentez et adaptez le chemin si nécessaire.
#
# import pandas as pd
#
# data_path = r"D:\chemin\vers\olist_dataset"
#
# orders = pd.read_csv(
#     Path(data_path) / "olist_orders_dataset.csv"
# )
#
# items = pd.read_csv(
#     Path(data_path) / "olist_order_items_dataset.csv"
# )
#
# products = pd.read_csv(
#     Path(data_path) / "olist_products_dataset.csv"
# )
#
# cat_trans = pd.read_csv(
#     Path(data_path) / "product_category_name_translation.csv"
# )
#
# sellers = pd.read_csv(
#     Path(data_path) / "olist_sellers_dataset.csv"
# )
#
# customers = pd.read_csv(
#     Path(data_path) / "olist_customers_dataset.csv"
# )
#
# reviews = pd.read_csv(
#     Path(data_path) / "olist_order_reviews_dataset.csv"
# )
#
# Cette méthode est commentée.
# La suite utilise kagglehub.

In [2]:
# ============================================================
# MÉTHODE 2 — CHARGEMENT AVEC KAGGLEHUB
# ============================================================

import os
import pandas as pd
import numpy as np
import kagglehub

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

print("Téléchargement / récupération du dataset Olist...")

dataset_path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce"
)

print(f"Dataset disponible dans : {dataset_path}")

files_mapping = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "cat_trans": "product_category_name_translation.csv",
    "sellers": "olist_sellers_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
}

tables = {}

for name, filename in files_mapping.items():
    filepath = os.path.join(dataset_path, filename)

    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"Fichier introuvable : {filepath}"
        )

    tables[name] = pd.read_csv(filepath)

orders = tables["orders"]
items = tables["items"]
products = tables["products"]
cat_trans = tables["cat_trans"]
sellers = tables["sellers"]
customers = tables["customers"]
reviews = tables["reviews"]

print("\n7 tables chargées :")

for name, df_table in tables.items():
    print(
        f"{name:<12} : "
        f"{len(df_table):>7,} lignes × "
        f"{df_table.shape[1]} colonnes"
    )

Téléchargement / récupération du dataset Olist...
Dataset disponible dans : C:\Users\Trezkit\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2

7 tables chargées :
orders       :  99,441 lignes × 8 colonnes
items        : 112,650 lignes × 7 colonnes
products     :  32,951 lignes × 9 colonnes
cat_trans    :      71 lignes × 2 colonnes
sellers      :   3,095 lignes × 4 colonnes
customers    :  99,441 lignes × 5 colonnes
reviews      :  99,224 lignes × 7 colonnes


# PARTIE 0 — Nouvelles notions

## 0.1 — Cardinalité

La cardinalité décrit la manière dont les lignes d'une table correspondent aux
lignes d'une autre table.

### Principaux cas

- **un-à-un** : une ligne correspond à une seule ligne ;
- **un-à-plusieurs** : une ligne peut correspondre à plusieurs lignes ;
- **plusieurs-à-plusieurs** : plusieurs lignes d'une table peuvent correspondre
  à plusieurs lignes de l'autre.

### Exemples Olist

`sellers → items`

Un vendeur peut réaliser plusieurs ventes.

La relation est donc :

> **un vendeur → plusieurs lignes de vente**

`orders → items`

Une commande peut contenir plusieurs produits.

La relation est donc :

> **une commande → plusieurs lignes de vente**

`orders → reviews`

Le sujet nous demande justement de vérifier la cardinalité réelle avant
d'utiliser cette relation.

### Pourquoi est-ce important ?

Une mauvaise hypothèse de cardinalité peut provoquer une multiplication des
lignes lors d'un `merge`.

Cette multiplication peut ensuite fausser :

- le chiffre d'affaires ;
- le nombre de ventes ;
- les moyennes ;
- les proportions ;
- les KPI.

## 0.2 — Format long et format large

### Format long

Une ligne représente une observation.

Exemple :

| catégorie | État | CA |
|---|---|---:|
| informatique | SP | 100000 |
| informatique | RJ | 50000 |
| meubles | SP | 70000 |

### Format large

Les valeurs d'une dimension deviennent des colonnes.

Exemple :

| catégorie | SP | RJ |
|---|---:|---:|
| informatique | 100000 | 50000 |
| meubles | 70000 | 0 |

Pandas travaille naturellement avec des données longues.

`pivot_table()` permet ensuite de construire un tableau large pour faciliter
la lecture et la comparaison.

## 0.3 — KPI

KPI signifie **Key Performance Indicator**.

Un KPI est une mesure liée à une question métier et susceptible d'aider à
prendre une décision.

Exemples :

- chiffre d'affaires ;
- nombre de ventes ;
- prix moyen ;
- score moyen de satisfaction ;
- proportion de ventes locales.

Un nombre calculé n'est donc pas automatiquement un KPI.

Le rôle de l'analyste est de sélectionner les indicateurs réellement utiles
pour le problème posé.

## 0.4 — Tendance et saisonnalité

### Tendance

Une tendance représente une évolution générale sur une période.

### Saisonnalité

Une saisonnalité correspond à un comportement qui se répète selon une période,
un cycle ou un événement.

Dans ce cas pratique, nous allons observer le chiffre d'affaires mensuel.

Mais un mois exceptionnel ne doit pas automatiquement être interprété comme
une tendance structurelle.

Nous devons notamment vérifier :

- les mois présents ;
- le premier mois ;
- le dernier mois ;
- la complétude des périodes.

## 0.5 — Feature engineering

Le feature engineering consiste à créer une nouvelle variable à partir des
variables existantes.

Exemple :

`price = 87.50`

peut devenir :

`gamme_prix = "Standard"`

L'Exercice 15 utilise `apply()` pour créer une variable commerciale.

# PARTIE EDA — Explorer avant de croiser

Avant le premier `merge`, chaque table doit être étudiée séparément.

Le but est de connaître :

- son volume ;
- ses colonnes ;
- ses types ;
- ses valeurs manquantes ;
- ses clés ;
- sa cardinalité ;
- ses particularités métier.

### Exercice 1 — Audit des 7 tables [8 pts]

Pour chaque table, produire :

- le nombre de lignes ;
- le nombre de colonnes ;
- le type de chaque colonne.

Une ligne du tableau final correspond à une table.

In [38]:
import pandas as pd

# ============================================================
# AUDIT STRUCTUREL DES TABLES
# ============================================================

audit_structure = []

for name, df_table in tables.items():

    # Comptage des différents types de données
    types = df_table.dtypes.astype(str).value_counts()

    types_detail = " | ".join(
        f"{dtype}: {count}"
        for dtype, count in types.items()
    )

    audit_structure.append({
        "Table": name,
        "Lignes": len(df_table),
        "Colonnes": df_table.shape[1],
        "Types": types_detail
    })

audit_structure = pd.DataFrame(audit_structure)


# ============================================================
# AFFICHAGE CONSOLE
# ============================================================

print("\n" + "=" * 90)
print("📐 AUDIT STRUCTUREL DES TABLES")
print("=" * 90)

print(
    audit_structure.to_string(
        index=False,
        justify="left"
    )
)

print("=" * 90)
print(f"📊 Nombre de tables auditées : {len(audit_structure)}")
print("=" * 90)


📐 AUDIT STRUCTUREL DES TABLES
Table      Lignes  Colonnes Types                         
   orders  99441  8                                 str: 8
    items 112650  7         str: 4 | float64: 2 | int64: 1
 products  32951  9                    float64: 7 | str: 2
cat_trans     71  2                                 str: 2
  sellers   3095  4                      str: 3 | int64: 1
customers  99441  5                      str: 4 | int64: 1
  reviews  99224  7                      str: 6 | int64: 1
📊 Nombre de tables auditées : 7


### Conclusion — Exercice 1

L'audit montre que les sept tables n'ont pas le même niveau de granularité :
`orders` et `customers` contiennent **99 441 lignes**, `items` **112 650**,
`products` **32 951**, `cat_trans` **71**, `sellers` **3 095** et `reviews`
**99 224**.

Les clés principales sont complètes et uniques pour `orders`, `products`,
`cat_trans`, `sellers` et `customers`. Deux points demandent une attention
particulière : `reviews` possède des `review_id` répétés et `items` ne peut pas
être contrôlée par `order_item_id` seul, car cette colonne est répétée entre
les commandes.

**Décision analytique :** avant chaque `merge`, nous devons utiliser la clé et
la cardinalité correspondant réellement au grain de la table.

## Exercice 2 — Valeurs manquantes [8 pts]

Pour chaque colonne des sept tables, calculer :

- le nombre de valeurs manquantes ;
- leur proportion.

Puis répondre aux deux questions métier :

1. Une colonne de texte libre dans `reviews` est-elle nécessairement
   défectueuse lorsqu'elle est largement vide ?
2. Pourquoi certaines dates de livraison dans `orders` peuvent-elles être
   absentes sans être nécessairement des erreurs ?

In [45]:
# ============================================================
# AUDIT GLOBAL DES VALEURS MANQUANTES
# ============================================================

missing_rows = []

for name, df_table in tables.items():

    for column in df_table.columns:

        missing_count = int(
            df_table[column].isna().sum()
        )

        proportion = (
            missing_count / len(df_table) * 100
            if len(df_table) > 0
            else 0
        )

        missing_rows.append({
            "table": name,
            "colonne": column,
            "valeurs_manquantes": missing_count,
            "proportion_pct": round(proportion, 2)
        })


# ============================================================
# CONSTRUCTION DU RÉSUMÉ
# ============================================================

missing_summary = (
    pd.DataFrame(missing_rows)
    .sort_values(
        ["valeurs_manquantes", "table"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


# ============================================================
# AFFICHAGE CONSOLE
# ============================================================

print("\n" + "═" * 95)
print("🔎  AUDIT GLOBAL DES VALEURS MANQUANTES")
print("═" * 95)

print(
    f"\n📊 Tables analysées        : {len(tables)}"
)

print(
    f"📋 Colonnes analysées     : {len(missing_summary):,}"
)

print(
    f"❌ Valeurs manquantes     : "
    f"{missing_summary['valeurs_manquantes'].sum():,}"
)

print("\n" + "─" * 95)

print(
    missing_summary.to_string(
        index=False,
        justify="left"
    )
)

print("\n" + "═" * 95)


═══════════════════════════════════════════════════════════════════════════════════════════════
🔎  AUDIT GLOBAL DES VALEURS MANQUANTES
═══════════════════════════════════════════════════════════════════════════════════════════════

📊 Tables analysées        : 7
📋 Colonnes analysées     : 42
❌ Valeurs manquantes     : 153,259

───────────────────────────────────────────────────────────────────────────────────────────────
table     colonne                        valeurs_manquantes  proportion_pct
  reviews          review_comment_title 87656               88.34          
  reviews        review_comment_message 58247               58.70          
   orders order_delivered_customer_date  2965                2.98          
   orders  order_delivered_carrier_date  1783                1.79          
 products         product_category_name   610                1.85          
 products           product_name_lenght   610                1.85          
 products    product_description_lenght   6

In [42]:
# ============================================================
# AUDIT DES DATES DE LIVRAISON — TABLE ORDERS
# ============================================================

delivery_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]


# ------------------------------------------------------------
# 1. Identification des commandes concernées
# ------------------------------------------------------------

orders_missing_delivery = orders[
    orders[delivery_columns].isna().any(axis=1)
].copy()


# ------------------------------------------------------------
# 2. Affichage du diagnostic
# ------------------------------------------------------------

print("\n" + "═" * 80)
print("🚚  AUDIT DES DATES DE LIVRAISON — ORDERS")
print("═" * 80)

print(
    f"\n📦 Commandes analysées        : {len(orders):,}"
)

print(
    f"⚠️ Commandes avec date manquante : "
    f"{len(orders_missing_delivery):,}"
)

proportion_missing = (
    len(orders_missing_delivery) / len(orders) * 100
    if len(orders) > 0
    else 0
)

print(
    f"📊 Proportion concernée       : "
    f"{proportion_missing:.2f}%"
)


# ------------------------------------------------------------
# 3. Répartition par statut
# ------------------------------------------------------------

status_summary = (
    orders_missing_delivery["order_status"]
    .value_counts()
    .rename_axis("Statut")
    .reset_index(name="Nombre de commandes")
)


print("\n" + "─" * 80)
print("📋  RÉPARTITION DES COMMANDES CONCERNÉES PAR STATUT")
print("─" * 80)

if status_summary.empty:

    print("✅ Aucune commande concernée.")

else:

    print(
        status_summary.to_string(
            index=False,
            justify="left"
        )
    )

print("\n" + "═" * 80)


════════════════════════════════════════════════════════════════════════════════
🚚  AUDIT DES DATES DE LIVRAISON — ORDERS
════════════════════════════════════════════════════════════════════════════════

📦 Commandes analysées        : 99,441
⚠️ Commandes avec date manquante : 2,980
📊 Proportion concernée       : 3.00%

────────────────────────────────────────────────────────────────────────────────
📋  RÉPARTITION DES COMMANDES CONCERNÉES PAR STATUT
────────────────────────────────────────────────────────────────────────────────
Statut       Nombre de commandes
    shipped 1107                
   canceled  619                
unavailable  609                
   invoiced  314                
 processing  301                
  delivered   23                
    created    5                
   approved    2                

════════════════════════════════════════════════════════════════════════════════



### Conclusion — Exercice 2

L’audit global met en évidence **153 259 valeurs manquantes** sur l’ensemble des tables analysées. Les valeurs manquantes sont principalement concentrées dans la table `reviews`, notamment dans les champs `review_comment_title` et `review_comment_message`.

Dans `reviews`, on compte **145 903 valeurs manquantes**. Le champ `review_comment_message` est absent dans **58 247 lignes (58,70 %)** et `review_comment_title` dans **87 656 lignes (88,34 %)**. L’absence d’un commentaire ou d’un titre ne signifie pas qu’un avis est invalide : ces champs peuvent être absents tout en conservant un `review_score` valide. Ils sont donc considérés comme des attributs facultatifs de l’avis.

Dans `orders`, les valeurs manquantes concernent principalement les dates liées au cycle de vie de la commande : `order_approved_at`, `order_delivered_carrier_date` et `order_delivered_customer_date`. Au total, **2 980 commandes**, soit **3,00 % des commandes**, présentent au moins une date manquante parmi ces trois champs.

La répartition par `order_status` montre que ces absences concernent principalement des commandes qui n'ont pas nécessairement atteint l'étape de livraison. Les statuts `shipped`, `canceled`, `unavailable`, `invoiced` et `processing` représentent notamment une part importante des commandes concernées. L’absence d’une date de livraison est donc souvent cohérente avec l’état de la commande.

Cependant, les cas de commandes portant le statut `delivered` doivent faire l’objet d’une vérification spécifique, car une commande livrée sans date de livraison client constitue une anomalie potentielle à documenter plutôt qu'une valeur à imputer automatiquement.

**Décision analytique :** aucune imputation automatique n'est appliquée aux dates de commande ni aux champs de commentaire. Les valeurs manquantes sont conservées afin de respecter l'information réellement présente dans les données. Les cas potentiellement incohérents seront identifiés séparément et documentés avant toute transformation.


## Exercice 3 — `review_id` est-il vraiment unique ? [8 pts]

Le sujet indique que `review_id` est censé identifier un avis.

Nous devons vérifier :

1. le nombre de `review_id` distincts ;
2. le nombre d'occurrences répétées ;
3. si un `review_id` répété correspond au même `order_id` ;
4. ou s'il est associé à plusieurs commandes.

Cette vérification est essentielle avant le `merge` de l'Exercice 7.

In [47]:
# ============================================================
# AUDIT DE L'UNICITÉ DE review_id
# ============================================================

review_id_distinct = reviews["review_id"].nunique()

review_id_repeated_occurrences = (
    reviews["review_id"].duplicated().sum()
)


# ============================================================
# IDENTIFICATION DES review_id RÉPÉTÉS
# ============================================================

duplicated_reviews = reviews[
    reviews["review_id"].duplicated(keep=False)
].copy()


# ============================================================
# ANALYSE DE LA CARDINALITÉ
# ============================================================

review_cardinality = (
    duplicated_reviews
    .groupby("review_id")
    .agg(
        nombre_lignes=("order_id", "size"),
        nombre_commandes=("order_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["nombre_lignes", "nombre_commandes"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)


# ============================================================
# NOMBRE DE review_id ASSOCIÉS À PLUSIEURS COMMANDES
# ============================================================

multiple_orders = (
    review_cardinality["nombre_commandes"] > 1
).sum()


# ============================================================
# AFFICHAGE CONSOLE
# ============================================================

print("\n" + "═" * 90)
print("🔑  AUDIT DE L'UNICITÉ DE review_id")
print("═" * 90)

print(
    f"\n📋 Nombre de lignes reviews          : {len(reviews):,}"
)

print(
    f"🔢 review_id distincts              : {review_id_distinct:,}"
)

print(
    f"🔁 Occurrences répétées après la première : "
    f"{review_id_repeated_occurrences:,}"
)

print(
    f"⚠️ review_id apparaissant plusieurs fois : "
    f"{len(review_cardinality):,}"
)

print(
    f"🔗 review_id associés à plusieurs commandes : "
    f"{multiple_orders:,}"
)


# ============================================================
# DÉTAIL DES review_id RÉPÉTÉS
# ============================================================

print("\n" + "─" * 90)
print("📊  DÉTAIL DES review_id RÉPÉTÉS — 20 PREMIERS CAS")
print("─" * 90)

if review_cardinality.empty:

    print("✅ Aucun review_id répété détecté.")

else:

    print(
        review_cardinality.head(20).to_string(
            index=False
        )
    )

print("\n" + "═" * 90)


══════════════════════════════════════════════════════════════════════════════════════════
🔑  AUDIT DE L'UNICITÉ DE review_id
══════════════════════════════════════════════════════════════════════════════════════════

📋 Nombre de lignes reviews          : 99,224
🔢 review_id distincts              : 98,410
🔁 Occurrences répétées après la première : 814
⚠️ review_id apparaissant plusieurs fois : 789
🔗 review_id associés à plusieurs commandes : 789

──────────────────────────────────────────────────────────────────────────────────────────
📊  DÉTAIL DES review_id RÉPÉTÉS — 20 PREMIERS CAS
──────────────────────────────────────────────────────────────────────────────────────────
                       review_id  nombre_lignes  nombre_commandes
08528f70f579f0c830189efc523d2182              3                 3
0c76e7a547a531e7bf9f0b99cba071c1              3                 3
1fb4ddc969e6bea80e38deec00393a6f              3                 3
2172867fd5b1a55f98fe4608e1547b4b              3     

In [48]:
# ============================================================
# CONTRÔLE RÉFÉRENTIEL : reviews → orders
# ============================================================

orders_ids = set(orders["order_id"].dropna())

reviews_without_order = reviews[
    ~reviews["order_id"].isin(orders_ids)
]

print("\n" + "─" * 90)
print("🔗  CONTRÔLE RÉFÉRENTIEL : reviews → orders")
print("─" * 90)

print(
    f"Reviews sans commande correspondante : "
    f"{len(reviews_without_order):,}"
)

if reviews_without_order.empty:
    print("✅ Toutes les reviews possèdent une commande correspondante.")
else:
    print("⚠️ Des reviews sans commande correspondante ont été détectées.")

print("═" * 90)


──────────────────────────────────────────────────────────────────────────────────────────
🔗  CONTRÔLE RÉFÉRENTIEL : reviews → orders
──────────────────────────────────────────────────────────────────────────────────────────
Reviews sans commande correspondante : 0
✅ Toutes les reviews possèdent une commande correspondante.
══════════════════════════════════════════════════════════════════════════════════════════


### Conclusion — Exercice 3

La table `reviews` contient **99 224 lignes** et **98 410 `review_id` distincts**. L’audit met en évidence **814 occurrences répétées après la première occurrence**, correspondant à **789 `review_id` distincts dupliqués**.

Le contrôle de cardinalité montre que les **789 `review_id` répétés sont associés à plusieurs `order_id` distincts**. Par conséquent, `review_id` ne peut pas être considéré comme une clé unique pour effectuer directement une jointure avec les données de ventes.

Le contrôle référentiel entre `reviews.order_id` et `orders.order_id` donne toutefois un résultat satisfaisant : **aucune review ne possède un `order_id` absent de la table `orders`**. Les **99 224 reviews** disposent donc d’une commande correspondante dans `orders`.

Cette distinction est importante pour la suite de l’analyse. Une jointure directe utilisant `review_id` comme clé unique pourrait entraîner une mauvaise interprétation de la structure des données. Plus généralement, une jointure avec `reviews` doit tenir compte de sa cardinalité afin d’éviter une **multiplication des lignes** et une éventuelle distorsion des KPI.

**Décision analytique :** `review_id` ne sera pas utilisé comme clé unique de rattachement aux données de ventes. Les informations de `reviews` seront **agrégées au niveau `order_id` avant la jointure**, afin de maîtriser la cardinalité, d’éviter la multiplication des lignes et de préserver la fiabilité des indicateurs calculés.


# PARTIE A — Construire le DataFrame enrichi avec `merge`

Nous partons de `items`.

Le grain initial est :

> **1 ligne = 1 produit vendu dans une commande.**

Nous allons progressivement enrichir ce DataFrame.

## Exercice 4 — Enrichir les ventes avec les produits [6 pts]

Relier :

`items → products`

via :

`product_id`

Objectif : ajouter notamment la catégorie du produit.

La jointure `left` permet de conserver toutes les lignes de vente.

In [ ]:
# ============================================================
# JOINTURE ITEMS → PRODUCTS
# Contrôle de cardinalité : many_to_one
# ============================================================

df = items.copy()

# Nombre de lignes avant la jointure
lignes_avant = len(df)


# ------------------------------------------------------------
# Jointure avec PRODUCTS
# ------------------------------------------------------------

df = df.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)


# Nombre de lignes après la jointure
lignes_apres = len(df)

variation = lignes_apres - lignes_avant


# ============================================================
# AUDIT DE LA JOINTURE
# ============================================================

print("\n" + "═" * 90)
print("🔗  AUDIT DE LA JOINTURE : ITEMS → PRODUCTS")
print("═" * 90)

print(
    f"\n📦 Lignes avant la jointure : {lignes_avant:,}"
)

print(
    f"📦 Lignes après la jointure  : {lignes_apres:,}"
)

print(
    f"📊 Variation                : {variation:+,}"
)


# ------------------------------------------------------------
# Vérification de conservation des lignes
# ------------------------------------------------------------

assert lignes_avant == lignes_apres, (
    "❌ Le nombre de lignes a changé après la jointure."
)

print(
    "\n✅ Contrôle de cardinalité : many_to_one respecté."
)

print(
    "✅ Aucune multiplication des lignes détectée."
)

print(
    "✅ Toutes les lignes de items sont conservées."
)


# ============================================================
# APERÇU DU DATAFRAME ENRICHI
# ============================================================

print("\n" + "─" * 90)
print("📋  APERÇU DES DONNÉES APRÈS ENRICHISSEMENT")
print("─" * 90)

display(
    df[
        [
            "order_id",
            "order_item_id",
            "product_id",
            "product_category_name",
            "price"
        ]
    ].head()
)

print("\n" + "═" * 90)

Lignes avant : 112650
Lignes après : 112650
Variation : 0


,order_id,order_item_id,product_id,product_category_name,price
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,pet_shop,239.90
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,perfumaria,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,199.90


### Conclusion — Exercice 4

La jointure `items → products` conserve les **112 650 lignes de vente** :
aucune ligne n'est perdue et aucune multiplication de lignes n'est observée.

Cette stabilité est cohérente avec la cardinalité attendue :

> plusieurs lignes de vente → un produit catalogue.

La clé `product_id` est unique dans `products`, ce qui confirme que la table
produits peut être utilisée comme table de référence pour enrichir les ventes.

**Décision analytique :** l'enrichissement par `product_id` est validé avec une
relation `many_to_one`.

## Exercice 5 — Traduire les catégories [8 pts]

Relier le DataFrame actuel à `cat_trans` via :

`product_category_name`

Nous comparons :

- `inner`
- `left`

L'objectif commercial est de ne perdre aucune vente.

In [51]:
df_inner_test = df.merge(
    cat_trans,
    on="product_category_name",
    how="inner"
)

df_left_test = df.merge(
    cat_trans,
    on="product_category_name",
    how="left"
)

print("Lignes dans df :", len(df))
print("Lignes après inner :", len(df_inner_test))
print("Lignes après left :", len(df_left_test))

print(
    "Ventes potentiellement exclues par inner :",
    len(df) - len(df_inner_test)
)
# ============================================================
# 👀  APERÇU DU DATAFRAME — HEAD & TAIL
# ============================================================

print("\n" + "═" * 100)
print("📊  APERÇU DU DATAFRAME : edf_left_test")
print("═" * 100)

# ------------------------------------------------------------
# 🔹 1. Premières lignes
# ------------------------------------------------------------

print("\n🔹 HEAD — 5 premières lignes")
print("─" * 100)

print(
    df_left_test
    .head()
    .to_string(index=False)
)

# ------------------------------------------------------------
# 🔹 2. Dernières lignes
# ------------------------------------------------------------

print("\n🔹 TAIL — 20 dernières lignes")
print("─" * 100)

print(
    df_left_test
    .tail(20)
    .to_string(index=False)
)

print("\n" + "═" * 100)
print(
    f"📌 Dimensions du DataFrame : "
    f"{df_left_test.shape[0]:,} lignes × {df_left_test.shape[1]} colonnes"
)
print("═" * 100)


Lignes dans df : 112650
Lignes après inner : 111023
Lignes après left : 112650
Ventes potentiellement exclues par inner : 1627

════════════════════════════════════════════════════════════════════════════════════════════════════
📊  APERÇU DU DATAFRAME : edf_left_test
════════════════════════════════════════════════════════════════════════════════════════════════════

🔹 HEAD — 5 premières lignes
────────────────────────────────────────────────────────────────────────────────────────────────────
                        order_id  order_item_id                       product_id                        seller_id shipping_limit_date  price  freight_value product_category_name  product_name_lenght  product_description_lenght  product_photos_qty  product_weight_g  product_length_cm  product_height_cm  product_width_cm product_category_name_english_x category_analysis   seller_city seller_state  review_score  review_count                      customer_id customer_state  vente_locale type_vente ga

### Conclusion — Exercice 5

Le contrôle de la catégorisation des produits met en évidence **623 produits sans correspondance de traduction exploitable** : **610 produits sans catégorie source** et **13 produits dont la catégorie source existe mais ne possède pas de correspondance dans la table de traduction**.

Parmi ces cas, les produits sans catégorie source sont effectivement présents dans les ventes. Ils concernent **1 603 lignes de commande**, représentant **179 535,28 BRL**, soit **1,32 % du montant total des ventes**.

Le test des jointures confirme également l'impact d'un `inner join` : sur **112 650 lignes de vente**, seulement **111 023** seraient conservées, soit **1 627 lignes perdues**. À l'inverse, le `left join` conserve les **112 650 lignes initiales**.

**Décision analytique :** la jointure finale doit donc rester en **`left`** afin de ne pas exclure des ventes simplement parce qu'une information de catégorisation ou de traduction est absente.

Les catégories disponibles sont conservées lorsqu'elles existent, tandis que les produits sans catégorie ou sans traduction restent identifiables pour un traitement ultérieur. Cette approche permet de **préserver l'intégralité des ventes tout en conservant la traçabilité des anomalies de catégorisation**.


## Exercice 6 — Localisation des vendeurs [6 pts]

Relier :

`df → sellers`

via :

`seller_id`

Objectif : récupérer :

- `seller_city`
- `seller_state`

In [53]:
# ============================================================
# 🏪  AJOUT DES INFORMATIONS VENDEUR
# ============================================================

# Nombre de lignes avant la jointure
lignes_avant = len(df)

# Jointure avec la table sellers
df = df.merge(
    sellers[
        [
            "seller_id",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left",
    validate="many_to_one"
)

# Vérification : aucune ligne ne doit être ajoutée ou supprimée
assert len(df) == lignes_avant

# Affichage du résultat
print("\n" + "═" * 90)
print("🏪  INFORMATIONS VENDEUR AJOUTÉES")
print("═" * 90)

print(f"\n📌 Lignes avant la jointure : {lignes_avant:,}")
print(f"📌 Lignes après la jointure : {len(df):,}")
print(f"📌 Variation               : {len(df) - lignes_avant:+,}")

print("\n🔹 Aperçu des informations vendeur")
print("─" * 90)

print(
    df[
        [
            "seller_id",
            "seller_city",
            "seller_state"
        ]
    ]
    .head()
    .to_string(index=False)
)

print("\n" + "═" * 90)
print("✅ Jointure validée : aucune modification du nombre de lignes.")
print("═" * 90)


══════════════════════════════════════════════════════════════════════════════════════════
🏪  INFORMATIONS VENDEUR AJOUTÉES
══════════════════════════════════════════════════════════════════════════════════════════

📌 Lignes avant la jointure : 112,650
📌 Lignes après la jointure : 112,650
📌 Variation               : +0

🔹 Aperçu des informations vendeur
──────────────────────────────────────────────────────────────────────────────────────────
                       seller_id   seller_city seller_state
48436dade18ac8b2bce089ec2a041202 volta redonda           SP
dd7ddc04e1b6c2c614352b383efe2d36     sao paulo           SP
5b51032eddd242adc84c38acab88f23d borda da mata           MG
9d7a1d34a5052409006425275ba1c2b4        franca           SP
df560393f3a51e74553ab94004ba5c87        loanda           PR

══════════════════════════════════════════════════════════════════════════════════════════
✅ Jointure validée : aucune modification du nombre de lignes.
══════════════════════════════════════

### Conclusion — Exercice 6

La table `sellers` contient **3 095 vendeurs**. L'audit préalable de la table confirme que `seller_id` constitue une clé unique et complète et que les informations géographiques contrôlées ne présentent pas de valeurs manquantes.

La jointure `df → sellers` a été réalisée sur **`seller_id`** avec une cardinalité **`many_to_one`** et une jointure **`left`**.

Le contrôle avant/après confirme que le nombre de lignes reste inchangé :

* **Avant la jointure : 112 650 lignes**
* **Après la jointure : 112 650 lignes**
* **Variation : 0 ligne**

La jointure permet ainsi d'ajouter `seller_city` et `seller_state` sans perte ni multiplication des lignes. Le grain initial du DataFrame est donc conservé : **une ligne correspond toujours à une ligne de commande**.

**Décision analytique :** la jointure `df → sellers` est validée. Les informations `seller_city` et `seller_state` peuvent être utilisées pour les analyses géographiques des ventes, sous réserve que l'audit de qualité de `sellers` reste conforme aux contrôles précédents.


## Exercice 7 — Ajouter la satisfaction [8 pts]

Nous voulons ajouter `review_score` à `df` via `order_id`.

Mais l'Exercice 3 a montré qu'il faut vérifier la cardinalité.

Nous allons donc effectuer volontairement le `merge` brut afin de mesurer son
impact sur le nombre de lignes.

In [ ]:
# ============================================================
# ⭐  AUDIT — JOINTURE DES AVIS CLIENTS
# ============================================================

# Nombre de lignes avant la jointure
lignes_avant_review = len(df)

# ------------------------------------------------------------
# 🔹 1. Jointure avec la table reviews
# ------------------------------------------------------------

df_review_raw = df.merge(
    reviews[
        [
            "order_id",
            "review_id",
            "review_score"
        ]
    ],
    on="order_id",
    how="left"
)

# Nombre de lignes après la jointure
lignes_apres_review = len(df_review_raw)

# Calcul de la variation
lignes_supplementaires = (
    lignes_apres_review - lignes_avant_review
)

# ------------------------------------------------------------
# 🔹 2. Résultats du contrôle
# ------------------------------------------------------------

print("\n" + "═" * 100)
print("⭐  AUDIT DE LA JOINTURE : df → reviews")
print("═" * 100)

print(f"\n📌 Lignes avant le merge reviews : {lignes_avant_review:,}")
print(f"📌 Lignes après le merge reviews : {lignes_apres_review:,}")
print(f"📌 Lignes supplémentaires        : {lignes_supplementaires:+,}")

# ------------------------------------------------------------
# 🔹 3. Interprétation automatique
# ------------------------------------------------------------

print("\n🔎 Interprétation")
print("─" * 100)

if lignes_supplementaires > 0:
    print(
        "⚠️  ATTENTION : la jointure a provoqué une multiplication "
        "des lignes."
    )
    print(
        "   → Plusieurs avis semblent être associés à certains "
        "order_id."
    )
    print(
        "   → Une agrégation des reviews par order_id est recommandée "
        "avant la jointure finale."
    )
else:
    print(
        "✅ Aucune multiplication de lignes détectée."
    )

print("\n" + "═" * 100)

Lignes avant le merge reviews : 112650
Lignes après le merge reviews : 113314
Lignes supplémentaires : 664


### Interprétation — Exercice 7

Le `merge` brut des reviews sur `order_id` montre pourquoi la cardinalité doit
être contrôlée avant de calculer des KPI.

Les reviews contiennent des `review_id` répétés et plusieurs commandes peuvent
être associées au même `review_id`. Une jointure directe peut donc multiplier
certaines lignes de vente.

Le brouillon confirme que **98 673 commandes distinctes** possèdent une
review, tandis que la table `reviews` contient **99 224 lignes**.

**Conclusion :** il serait dangereux de calculer le chiffre d'affaires après
un merge brut avec `reviews`. Nous agrégeons donc les reviews au niveau
`order_id` avant de les rattacher au DataFrame analytique.

In [55]:
# ============================================================
# ⭐  AUDIT — JOINTURE DES AVIS CLIENTS
# ============================================================

# Nombre de lignes avant la jointure
lignes_avant_review = len(df)

# ------------------------------------------------------------
# 🔹 1. Jointure avec la table reviews
# ------------------------------------------------------------

df_review_raw = df.merge(
    reviews[
        [
            "order_id",
            "review_id",
            "review_score"
        ]
    ],
    on="order_id",
    how="left"
)

# Nombre de lignes après la jointure
lignes_apres_review = len(df_review_raw)

# Calcul de la variation
lignes_supplementaires = (
    lignes_apres_review - lignes_avant_review
)

# ------------------------------------------------------------
# 🔹 2. Résultats du contrôle
# ------------------------------------------------------------

print("\n" + "═" * 100)
print("⭐  AUDIT DE LA JOINTURE : df → reviews")
print("═" * 100)

print(f"\n📌 Lignes avant le merge reviews : {lignes_avant_review:,}")
print(f"📌 Lignes après le merge reviews : {lignes_apres_review:,}")
print(f"📌 Lignes supplémentaires        : {lignes_supplementaires:+,}")

# ------------------------------------------------------------
# 🔹 3. Interprétation automatique
# ------------------------------------------------------------

print("\n🔎 Interprétation")
print("─" * 100)

if lignes_supplementaires > 0:
    print(
        "⚠️  ATTENTION : la jointure a provoqué une multiplication "
        "des lignes."
    )
    print(
        "   → Plusieurs avis semblent être associés à certains "
        "order_id."
    )
    print(
        "   → Une agrégation des reviews par order_id est recommandée "
        "avant la jointure finale."
    )
else:
    print(
        "✅ Aucune multiplication de lignes détectée."
    )

print("\n" + "═" * 100)


════════════════════════════════════════════════════════════════════════════════════════════════════
⭐  AUDIT DE LA JOINTURE : df → reviews
════════════════════════════════════════════════════════════════════════════════════════════════════

📌 Lignes avant le merge reviews : 112,650
📌 Lignes après le merge reviews : 113,314
📌 Lignes supplémentaires        : +664

🔎 Interprétation
────────────────────────────────────────────────────────────────────────────────────────────────────
⚠️  ATTENTION : la jointure a provoqué une multiplication des lignes.
   → Plusieurs avis semblent être associés à certains order_id.
   → Une agrégation des reviews par order_id est recommandée avant la jointure finale.

════════════════════════════════════════════════════════════════════════════════════════════════════


### Décision analytique — Exercice 7

Le test de jointure directe entre `df` et `reviews` sur `order_id` montre une **augmentation de 112 650 à 113 314 lignes**, soit **664 lignes supplémentaires**.

Cette augmentation confirme que plusieurs informations de review peuvent être associées à une même commande. Une jointure directe risquerait donc de **multiplier les lignes de vente** et de fausser certains indicateurs, notamment le chiffre d'affaires, le nombre d'articles ou le panier moyen.

La stratégie retenue consiste donc à **agréger les reviews au niveau de la commande (`order_id`)** :

* `review_score` → moyenne des scores associés à la commande ;
* `review_count` → nombre de reviews associées à la commande.

Après agrégation, chaque `order_id` doit apparaître **une seule fois** dans la table des reviews agrégées. La jointure avec `df` devient ainsi une relation **many-to-one**.

**Décision analytique :** les reviews seront agrégées avant la jointure finale afin de préserver le grain des ventes et d'intégrer les informations de satisfaction **sans provoquer de duplication des lignes ni de double comptage des indicateurs commerciaux**.


## Exercice 8 — Ajouter les clients et leur État [6 pts]

Nous devons réaliser deux fusions :

1. `df → orders` via `order_id` pour récupérer `customer_id` ;
2. `df → customers` via `customer_id` pour récupérer `customer_state`.

In [62]:

# ============================================================
# 🗺️  PARTIE A — ENRICHISSEMENT DES INFORMATIONS CLIENT
# ============================================================

print("\n" + "═" * 100)
print("🗺️  PARTIE A — ENRICHISSEMENT DES INFORMATIONS CLIENT")
print("═" * 100)


# ---------------------------------------------------------------------
# 1️⃣ Point de départ propre
# ---------------------------------------------------------------------
# On repart de items pour éviter les anciennes colonnes :
# customer_id_x, customer_id_y, etc.
#
# À ce stade, df représente une ligne de commande / order item.
# ---------------------------------------------------------------------

df = items.copy()

print(f"\n📦 Point de départ : {len(df):,} lignes")


# ---------------------------------------------------------------------
# 2️⃣ Enrichissement avec les produits
# ---------------------------------------------------------------------

lignes_avant = len(df)

df = df.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

lignes_apres = len(df)

assert lignes_avant == lignes_apres, (
    "❌ Le nombre de lignes a changé après la jointure products."
)

print("\n🛍️ Enrichissement produits")
print(f"📌 Lignes avant : {lignes_avant:,}")
print(f"📌 Lignes après : {lignes_apres:,}")
print(f"📌 Variation    : {lignes_apres - lignes_avant:+,}")


# ---------------------------------------------------------------------
# 3️⃣ Enrichissement avec les vendeurs
# ---------------------------------------------------------------------

lignes_avant = len(df)

df = df.merge(
    sellers[
        [
            "seller_id",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left",
    validate="many_to_one"
)

lignes_apres = len(df)

assert lignes_avant == lignes_apres, (
    "❌ Le nombre de lignes a changé après la jointure sellers."
)

print("\n🏪 Enrichissement vendeurs")
print(f"📌 Lignes avant : {lignes_avant:,}")
print(f"📌 Lignes après : {lignes_apres:,}")
print(f"📌 Variation    : {lignes_apres - lignes_avant:+,}")


# ---------------------------------------------------------------------
# 4️⃣ Récupération du client associé à chaque commande
# ---------------------------------------------------------------------
# orders possède la relation :
#
# order_id → customer_id
#
# Plusieurs lignes de items peuvent appartenir à une même commande.
# C'est donc une jointure many_to_one.
# ---------------------------------------------------------------------

lignes_avant = len(df)

df = df.merge(
    orders[
        [
            "order_id",
            "customer_id"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)

lignes_apres = len(df)

assert lignes_avant == lignes_apres, (
    "❌ Le nombre de lignes a changé après la jointure orders."
)

print("\n👤 Enrichissement avec l'identifiant client")
print(f"📌 Lignes avant : {lignes_avant:,}")
print(f"📌 Lignes après : {lignes_apres:,}")
print(f"📌 Variation    : {lignes_apres - lignes_avant:+,}")


# ---------------------------------------------------------------------
# 5️⃣ Récupération de l'État du client
# ---------------------------------------------------------------------
# customers possède :
#
# customer_id → customer_state
#
# customer_id est une clé unique dans customers.
# ---------------------------------------------------------------------

lignes_avant = len(df)

df = df.merge(
    customers[
        [
            "customer_id",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

lignes_apres = len(df)

assert lignes_avant == lignes_apres, (
    "❌ Le nombre de lignes a changé après la jointure customers."
)

print("\n🌎 Enrichissement avec la localisation client")
print(f"📌 Lignes avant : {lignes_avant:,}")
print(f"📌 Lignes après : {lignes_apres:,}")
print(f"📌 Variation    : {lignes_apres - lignes_avant:+,}")


# ---------------------------------------------------------------------
# 6️⃣ Contrôle final
# ---------------------------------------------------------------------

print("\n" + "─" * 100)
print("🔎 CONTRÔLE FINAL — PARTIE A")
print("─" * 100)

print(f"📊 Nombre final de lignes : {len(df):,}")

print("\n📋 Aperçu des données enrichies :")

print(
    df[
        [
            "order_id",
            "seller_state",
            "customer_id",
            "customer_state"
        ]
    ]
    .head()
    .to_string(index=False)
)

print("\n" + "═" * 100)
print("✅ PARTIE A TERMINÉE AVEC SUCCÈS")
print("═" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
🗺️  PARTIE A — ENRICHISSEMENT DES INFORMATIONS CLIENT
════════════════════════════════════════════════════════════════════════════════════════════════════

📦 Point de départ : 112,650 lignes

🛍️ Enrichissement produits
📌 Lignes avant : 112,650
📌 Lignes après : 112,650
📌 Variation    : +0

🏪 Enrichissement vendeurs
📌 Lignes avant : 112,650
📌 Lignes après : 112,650
📌 Variation    : +0

👤 Enrichissement avec l'identifiant client
📌 Lignes avant : 112,650
📌 Lignes après : 112,650
📌 Variation    : +0

🌎 Enrichissement avec la localisation client
📌 Lignes avant : 112,650
📌 Lignes après : 112,650
📌 Variation    : +0

────────────────────────────────────────────────────────────────────────────────────────────────────
🔎 CONTRÔLE FINAL — PARTIE A
────────────────────────────────────────────────────────────────────────────────────────────────────
📊 Nombre final de lignes : 112,650

📋 Aperçu des do

### Conclusion — Exercice 8

L’enrichissement `orders → customers` permet d’associer à chaque ligne de vente le `customer_id` puis le `customer_state`, sans modifier le grain du DataFrame.

Les contrôles effectués montrent que le nombre de lignes reste strictement stable : **112 650 lignes avant et après l’enrichissement**. Les jointures `orders` et `customers` respectent également une relation `many_to_one`, ce qui confirme qu’elles n’ont pas provoqué de multiplication artificielle des lignes.

La table `orders` contient **99 441 commandes**, tandis que `customers` permet de retrouver les informations géographiques associées aux clients. Le `customer_id` constitue ici la clé technique utilisée pour relier les différentes tables.

Il faut toutefois distinguer `customer_id` de `customer_unique_id`. Un même client réel peut être associé à plusieurs `customer_id`. Pour une analyse portant sur le **nombre de clients réellement distincts**, `customer_unique_id` est donc plus approprié.

**Décision analytique :** `customer_id` est utilisé comme clé technique pour les jointures, tandis que `customer_unique_id` doit être privilégié pour mesurer la clientèle réellement distincte. L’enrichissement réalisé conserve ainsi le grain des ventes tout en ajoutant les informations nécessaires aux futures analyses géographiques et clients.


In [66]:

# ============================================================
# 🔎 CONTRÔLE DE COHÉRENCE — PARTIE A
# ============================================================

print("\n" + "═" * 100)
print("🔎 CONTRÔLE DE COHÉRENCE — PARTIE A")
print("═" * 100)


# ---------------------------------------------------------------------
# 1️⃣ Colonnes attendues après l'enrichissement client
# ---------------------------------------------------------------------

expected_columns = [
    "order_id",
    "order_item_id",
    "product_id",
    "product_category_name",
    "seller_id",
    "seller_state",
    "customer_id",
    "customer_state",
    "price"
]


# ---------------------------------------------------------------------
# 2️⃣ Recherche des colonnes absentes
# ---------------------------------------------------------------------

missing_expected = [
    column
    for column in expected_columns
    if column not in df.columns
]


# ---------------------------------------------------------------------
# 3️⃣ Validation
# ---------------------------------------------------------------------

assert not missing_expected, (
    f"❌ Colonnes absentes : {missing_expected}"
)


# ---------------------------------------------------------------------
# 4️⃣ Contrôle du nombre de lignes
# ---------------------------------------------------------------------

expected_rows = 112_650

assert len(df) == expected_rows, (
    f"❌ Nombre de lignes inattendu : "
    f"{len(df):,} au lieu de {expected_rows:,}"
)


# ---------------------------------------------------------------------
# 5️⃣ Résultat
# ---------------------------------------------------------------------

print(f"📊 Shape de df : {df.shape}")
print(f"📌 Nombre de lignes : {len(df):,}")
print(f"📌 Nombre de colonnes : {len(df.columns)}")

print("\n📋 Colonnes client disponibles :")
print(
    df[
        [
            "order_id",
            "seller_state",
            "customer_id",
            "customer_state",
            "price"
        ]
    ]
    .head()
    .to_string(index=False)
)

print("\n" + "─" * 100)
print("✅ Toutes les colonnes nécessaires à la Partie A sont présentes.")
print("✅ Le nombre de lignes reste égal à 112 650.")
print("✅ CONTRÔLE PARTIE A : OK")
print("─" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
🔎 CONTRÔLE DE COHÉRENCE — PARTIE A
════════════════════════════════════════════════════════════════════════════════════════════════════
📊 Shape de df : (112650, 19)
📌 Nombre de lignes : 112,650
📌 Nombre de colonnes : 19

📋 Colonnes client disponibles :
                        order_id seller_state                      customer_id customer_state  price
00010242fe8c5a6d1ba2dd792cb16214           SP 3ce436f183e68e07877b285a838db11a             RJ  58.90
00018f77f2f0320c557190d7a144bdd3           SP f6dd3ec061db4e3987629fe6b26e5cce             SP 239.90
000229ec398224ef6ca0657da4fc703e           MG 6489ae5e4333f3693df5ad4372dab6d3             MG 199.00
00024acbcdf0a6daa1e931b038114c75           SP d4eb9395c8c0431ee92fce09860c5a06             SP  12.99
00042b26cf59d7ce69dfabb4e55b4fd9           PR 58dbd0b2d70206bf40e62cd34e84d795             SP 199.90

──────────────────────────────────────

# PARTIE B — Agréger avec `groupby`

Nous allons maintenant répondre aux principales questions de performance
commerciale.

## Exercice 9 — Top 10 des catégories par chiffre d'affaires [6 pts]

Calculer le chiffre d'affaires par catégorie.

### Définition utilisée

Ici :

> `chiffre_affaires = somme(price)`

Le fret n'est pas inclus dans cet indicateur.

In [76]:

# ============================================================
# 💰 EXERCICE 9 — TOP 10 DES CATÉGORIES PAR CHIFFRE D'AFFAIRES
# ============================================================

print("\n" + "═" * 100)
print("💰 EXERCICE 9 — TOP 10 DES CATÉGORIES PAR CHIFFRE D'AFFAIRES")
print("═" * 100)


# ---------------------------------------------------------------------
# 1️⃣ Vérification des colonnes nécessaires
# ---------------------------------------------------------------------

required_columns = [
    "product_category_name_english",
    "price"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

assert not missing_columns, (
    f"❌ Colonnes absentes : {missing_columns}"
)


# ---------------------------------------------------------------------
# 2️⃣ Calcul du chiffre d'affaires par catégorie
# ---------------------------------------------------------------------

top_10_categories = (
    df.groupby(
        "product_category_name_english",
        as_index=False
    )
    .agg(
        chiffre_affaires=("price", "sum")
    )
    .sort_values(
        "chiffre_affaires",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# 3️⃣ Affichage du résultat
# ---------------------------------------------------------------------

print("\n📊 TOP 10 DES CATÉGORIES PAR CHIFFRE D'AFFAIRES\n")

print(
    top_10_categories.to_string(index=False)
)


# ---------------------------------------------------------------------
# 4️⃣ Contrôle
# ---------------------------------------------------------------------

assert len(top_10_categories) <= 10, (
    "❌ Le résultat contient plus de 10 catégories."
)

assert top_10_categories["chiffre_affaires"].is_monotonic_decreasing, (
    "❌ Les catégories ne sont pas correctement triées."
)


print("\n" + "─" * 100)
print("✅ EXERCICE 9 TERMINÉ")
print("📌 Chiffre d'affaires = somme(price)")
print("📌 Freight non inclus")
print("─" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
💰 EXERCICE 9 — TOP 10 DES CATÉGORIES PAR CHIFFRE D'AFFAIRES
════════════════════════════════════════════════════════════════════════════════════════════════════

📊 TOP 10 DES CATÉGORIES PAR CHIFFRE D'AFFAIRES

product_category_name_english  chiffre_affaires
                health_beauty        1258681.34
                watches_gifts        1205005.68
               bed_bath_table        1036988.68
               sports_leisure         988048.97
        computers_accessories         911954.32
              furniture_decor         729762.49
                   cool_stuff         635290.85
                   housewares         632248.66
                         auto         592720.11
                 garden_tools         485256.46

────────────────────────────────────────────────────────────────────────────────────────────────────
✅ EXERCICE 9 TERMINÉ
📌 Chiffre d'affaires = somme(price)
📌

### Conclusion — Exercice 9

Le calcul du chiffre d'affaires par catégorie, selon la définition retenue **`chiffre_affaires = somme(price)`**, permet d'identifier les catégories présentant les montants de ventes de produits les plus élevés.

Les cinq premières catégories sont :

1. `health_beauty` : **1 258 681,34 BRL**
2. `watches_gifts` : **1 205 005,68 BRL**
3. `bed_bath_table` : **1 036 988,68 BRL**
4. `sports_leisure` : **988 048,97 BRL**
5. `computers_accessories` : **911 954,32 BRL**

La catégorie `health_beauty` présente le montant de chiffre d'affaires produit le plus élevé dans cet échantillon, avec **1 258 681,34 BRL**.

**Interprétation métier :** ces catégories représentent des pôles importants de chiffre d'affaires produit. Elles pourront donc être approfondies dans les analyses suivantes, notamment en étudiant leur performance selon les vendeurs, les régions et les niveaux de satisfaction client.

**Périmètre de l'indicateur :** le chiffre d'affaires présenté correspond exclusivement à la somme de `price`. Les frais de livraison (`freight_value`) ne sont pas inclus.


## Exercice 10 — Chiffre d'affaires par État vendeur [6 pts]

Calculer le chiffre d'affaires total pour chaque `seller_state`.

In [77]:

# ============================================================
# 🗺️ ANALYSE — CHIFFRE D'AFFAIRES PAR ÉTAT DU VENDEUR
# ============================================================

print("\n" + "═" * 100)
print("🗺️  CHIFFRE D'AFFAIRES PAR ÉTAT DU VENDEUR")
print("═" * 100)


# ---------------------------------------------------------------------
# 1️⃣ Vérification des colonnes nécessaires
# ---------------------------------------------------------------------

required_columns = [
    "seller_state",
    "price"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

assert not missing_columns, (
    f"❌ Colonnes absentes : {missing_columns}"
)


# ---------------------------------------------------------------------
# 2️⃣ Calcul du chiffre d'affaires par État
# ---------------------------------------------------------------------

revenue_by_state = (
    df.groupby(
        "seller_state",
        as_index=False
    )
    .agg(
        chiffre_affaires=("price", "sum")
    )
    .sort_values(
        "chiffre_affaires",
        ascending=False
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# 3️⃣ Affichage
# ---------------------------------------------------------------------

print("\n📊 Chiffre d'affaires produit par État du vendeur :\n")

print(
    revenue_by_state.to_string(index=False)
)


# ---------------------------------------------------------------------
# 4️⃣ Contrôles
# ---------------------------------------------------------------------

assert not revenue_by_state.empty, (
    "❌ Le résultat est vide."
)

assert revenue_by_state["chiffre_affaires"].is_monotonic_decreasing, (
    "❌ Les États ne sont pas correctement triés."
)


print("\n" + "─" * 100)
print("✅ Analyse du chiffre d'affaires par État terminée.")
print("📌 Chiffre d'affaires = somme(price)")
print("📌 Freight non inclus")
print("─" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
🗺️  CHIFFRE D'AFFAIRES PAR ÉTAT DU VENDEUR
════════════════════════════════════════════════════════════════════════════════════════════════════

📊 Chiffre d'affaires produit par État du vendeur :

seller_state  chiffre_affaires
          SP        8753396.21
          PR        1261887.21
          MG        1011564.74
          RJ         843984.22
          SC         632426.07
          RS         378559.54
          BA         285561.56
          DF          97749.48
          PE          91493.85
          GO          66399.21
          ES          47689.61
          MA          36408.95
          CE          20240.64
          PB          17095.00
          MT          17070.72
          RN           9992.60
          MS           8551.69
          RO           4762.20
          PI           2522.00
          SE           1606.20
          PA           1238.00
          AM       

### Conclusion — Exercice 10

L'analyse du chiffre d'affaires par État du vendeur montre une répartition géographique très inégale des ventes de produits.

Les cinq États associés aux montants de chiffre d'affaires les plus élevés sont :

* **SP : 8 753 396,21 BRL**
* **PR : 1 261 887,21 BRL**
* **MG : 1 011 564,74 BRL**
* **RJ : 843 984,22 BRL**
* **SC : 632 426,07 BRL**

L'État **SP (São Paulo)** présente de loin le montant de chiffre d'affaires le plus élevé dans les données analysées, avec **8 753 396,21 BRL**.

**Interprétation métier :** cette répartition met en évidence l'importance de la dimension géographique dans l'analyse de la performance des vendeurs. Les États présentant des montants plus élevés pourront être étudiés plus en détail selon les catégories de produits, les vendeurs et la satisfaction client.

**Périmètre de l'indicateur :** le chiffre d'affaires correspond à la **somme de `price`** par État du vendeur. Les frais de livraison (`freight_value`) ne sont pas inclus.


## Exercice 11 — Performance des 3 095 vendeurs [8 pts]

Pour chaque vendeur, calculer :

- nombre de ventes ;
- chiffre d'affaires ;
- prix moyen.

Puis classer les vendeurs par chiffre d'affaires et afficher les cinq premiers.

In [78]:

# ============================================================
# 🏪 EXERCICE — PERFORMANCE DES VENDEURS
# ============================================================

print("\n" + "═" * 100)
print("🏪 PERFORMANCE DES VENDEURS")
print("═" * 100)

# ---------------------------------------------------------------------
# 1️⃣ Agrégation des performances par vendeur
# ---------------------------------------------------------------------
seller_performance = (
    df.groupby(
        "seller_id",
        as_index=False
    )
    .agg(
        nombre_ventes=("price", "count"),
        chiffre_affaires=("price", "sum"),
        prix_moyen=("price", "mean")
    )
    .sort_values(
        "chiffre_affaires",
        ascending=False
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------
# 2️⃣ Sélection des 5 vendeurs générant le plus de chiffre d'affaires
# ---------------------------------------------------------------------
top_5_sellers = seller_performance.head(5)

# ---------------------------------------------------------------------
# 3️⃣ Contrôles
# ---------------------------------------------------------------------
print(
    f"📌 Nombre de vendeurs dans la table de performance : "
    f"{len(seller_performance):,}"
)

print(
    f"📌 Nombre de vendeurs affichés dans le Top 5      : "
    f"{len(top_5_sellers):,}"
)

# ---------------------------------------------------------------------
# 4️⃣ Affichage du Top 5
# ---------------------------------------------------------------------
print("\n🏆 TOP 5 DES VENDEURS PAR CHIFFRE D'AFFAIRES")
print("─" * 100)

print(
    top_5_sellers[
        [
            "seller_id",
            "nombre_ventes",
            "chiffre_affaires",
            "prix_moyen"
        ]
    ].to_string(index=False)
)

print("\n" + "═" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
🏪 PERFORMANCE DES VENDEURS
════════════════════════════════════════════════════════════════════════════════════════════════════
📌 Nombre de vendeurs dans la table de performance : 3,095
📌 Nombre de vendeurs affichés dans le Top 5      : 5

🏆 TOP 5 DES VENDEURS PAR CHIFFRE D'AFFAIRES
────────────────────────────────────────────────────────────────────────────────────────────────────
                       seller_id  nombre_ventes  chiffre_affaires  prix_moyen
4869f7a5dfa277a7dca6462dcf3b52b2           1156         229472.63  198.505735
53243585a1d6dc2643021fd1853d8905            410         222776.05  543.356220
4a3ca9315b744ce9f8e9374361493884           1987         200472.92  100.892260
fa1c13f2614d7b5c4749cbc52fecda94            586         194042.03  331.129744
7c67e1448b00f6e969d365cea6b010ab           1364         187923.89  137.774113

════════════════════════════════════════════

### Conclusion — Exercice 11

Les trois indicateurs calculés apportent des informations complémentaires sur la performance des vendeurs :

| Vendeur     | Nombre de ventes |   CA (BRL) | Prix moyen (BRL) |
| ----------- | ---------------: | ---------: | ---------------: |
| `4869f7...` |            1 156 | 229 472,63 |           198,51 |
| `532435...` |              410 | 222 776,05 |           543,36 |
| `4a3ca9...` |            1 987 | 200 472,92 |           100,89 |
| `fa1c13...` |              586 | 194 042,03 |           331,13 |
| `7c67e1...` |            1 364 | 187 923,89 |           137,77 |

Le classement par chiffre d'affaires ne correspond pas nécessairement au classement par nombre de ventes. Parmi ces cinq vendeurs, `4a3ca9...` présente le plus grand nombre de lignes de vente (1 987), tandis que `532435...` présente le prix moyen le plus élevé (543,36 BRL).

**Conclusion :** le chiffre d'affaires doit être analysé conjointement avec le nombre de ventes et le prix moyen afin de mieux comprendre le profil commercial des vendeurs. Un vendeur peut générer un CA élevé grâce à un volume important de ventes, à un prix moyen élevé, ou à une combinaison des deux.

> **Périmètre de l'analyse :** le chiffre d'affaires correspond à la somme de `price`. Le `freight_value` n'est pas inclus dans cet indicateur.


## Exercice 12 — Ventes locales ou nationales ? [8 pts]

Définition :

> Une vente est locale si `seller_state == customer_state`.

Sinon elle est nationale.

Nous allons calculer :

- le nombre de ventes locales ;
- le nombre de ventes nationales ;
- leur proportion.

In [79]:

# ============================================================
# 🗺️ EXERCICE — VENTES LOCALES VS NATIONALES
# ============================================================

print("\n" + "═" * 100)
print("🗺️  ANALYSE DES VENTES LOCALES ET NATIONALES")
print("═" * 100)

# ---------------------------------------------------------------------
# 1️⃣ Identification des ventes locales
# ---------------------------------------------------------------------
# Une vente est considérée comme locale lorsque le vendeur
# et le client se trouvent dans le même État.
df["vente_locale"] = (
    df["seller_state"] == df["customer_state"]
)

# ---------------------------------------------------------------------
# 2️⃣ Classification du type de vente
# ---------------------------------------------------------------------
df["type_vente"] = np.where(
    df["vente_locale"],
    "Locale",
    "Nationale"
)

# ---------------------------------------------------------------------
# 3️⃣ Calcul du nombre de ventes par type
# ---------------------------------------------------------------------
vente_locale_summary = (
    df["type_vente"]
    .value_counts()
    .rename_axis("type_vente")
    .reset_index(name="nombre_ventes")
)

# ---------------------------------------------------------------------
# 4️⃣ Calcul de la proportion de chaque type de vente
# ---------------------------------------------------------------------
vente_locale_summary["proportion_pct"] = (
    vente_locale_summary["nombre_ventes"]
    / vente_locale_summary["nombre_ventes"].sum()
    * 100
)

# ---------------------------------------------------------------------
# 5️⃣ Vérification de cohérence
# ---------------------------------------------------------------------
assert (
    vente_locale_summary["nombre_ventes"].sum()
    == len(df)
), "❌ Le total des ventes ne correspond pas au nombre de lignes de df."

assert (
    vente_locale_summary["proportion_pct"].sum()
    - 100
).round(10) == 0, "❌ Les proportions ne totalisent pas 100 %."

# ---------------------------------------------------------------------
# 6️⃣ Affichage du résumé
# ---------------------------------------------------------------------
print("\n📊 RÉPARTITION DES VENTES")
print("─" * 100)

print(
    vente_locale_summary.to_string(
        index=False,
        formatters={
            "proportion_pct": "{:.2f}%".format
        }
    )
)

# ---------------------------------------------------------------------
# 7️⃣ Calcul de la proportion des ventes locales
# ---------------------------------------------------------------------
proportion_locale = (
    df["vente_locale"].mean() * 100
)

print("\n📌 INDICATEUR CLÉ")
print("─" * 100)
print(
    f"Proportion de ventes locales : "
    f"{proportion_locale:.2f}%"
)

print("\n" + "═" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
🗺️  ANALYSE DES VENTES LOCALES ET NATIONALES
════════════════════════════════════════════════════════════════════════════════════════════════════

📊 RÉPARTITION DES VENTES
────────────────────────────────────────────────────────────────────────────────────────────────────
type_vente  nombre_ventes proportion_pct
 Nationale          71894         63.82%
    Locale          40756         36.18%

📌 INDICATEUR CLÉ
────────────────────────────────────────────────────────────────────────────────────────────────────
Proportion de ventes locales : 36.18%

════════════════════════════════════════════════════════════════════════════════════════════════════


### Conclusion — Exercice 12

Sur les **112 650 lignes de vente** analysées, **40 756** correspondent à des ventes locales, soit **36,18 %**, tandis que **71 894** correspondent à des ventes nationales, soit **63,82 %**.

La définition utilisée est volontairement simple :

> **Vente locale = `seller_state == customer_state`**

Cet indicateur mesure donc une proximité commerciale au **niveau de l'État**, et non une distance géographique réelle entre le vendeur et le client.

**Interprétation métier :** dans cet échantillon, les lignes de vente nationales représentent la majorité (**63,82 %**). La dimension géographique constitue ainsi un axe d'analyse pertinent et peut être mise en relation avec le chiffre d'affaires, les catégories de produits, les vendeurs et les indicateurs de satisfaction.


# PARTIE C — Construire des tableaux croisés avec `pivot_table`

## Exercice 13 — Catégories × États vendeurs [10 pts]

Construire une table croisée contenant :

- les 6 catégories avec le plus de CA ;
- les 5 États vendeurs avec le plus de CA ;
- le CA pour chaque combinaison.

Cette table permettra d'identifier les combinaisons catégorie / territoire
qui concentrent la valeur commerciale.

In [80]:

# ============================================================
# 📊 EXERCICE — CA PAR CATÉGORIE ET ÉTAT DU VENDEUR
# ============================================================

print("\n" + "═" * 100)
print("📊 CHIFFRE D'AFFAIRES — CATÉGORIES × ÉTATS VENDEURS")
print("═" * 100)

# ---------------------------------------------------------------------
# 1️⃣ Vérification des colonnes nécessaires
# ---------------------------------------------------------------------
colonnes_requises = [
    "product_category_name",
    "product_category_name_english",
    "seller_state",
    "price"
]

colonnes_manquantes = [
    col for col in colonnes_requises
    if col not in df.columns
]

if colonnes_manquantes:
    raise KeyError(
        "❌ Colonnes manquantes dans df : "
        + ", ".join(colonnes_manquantes)
    )

# ---------------------------------------------------------------------
# 2️⃣ Sélection des 6 catégories ayant le CA le plus élevé
# ---------------------------------------------------------------------
top_6_categories = (
    df.groupby("product_category_name_english")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(6)
    .index
)

# ---------------------------------------------------------------------
# 3️⃣ Sélection des 5 États vendeurs ayant le CA le plus élevé
# ---------------------------------------------------------------------
top_5_states = (
    df.groupby("seller_state")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
)

# ---------------------------------------------------------------------
# 4️⃣ Filtrage des données
# ---------------------------------------------------------------------
df_pivot = df[
    df["product_category_name_english"].isin(top_6_categories)
    & df["seller_state"].isin(top_5_states)
].copy()

# ---------------------------------------------------------------------
# 5️⃣ Construction du tableau croisé dynamique
# ---------------------------------------------------------------------
pivot_category_state = pd.pivot_table(
    df_pivot,
    values="price",
    index="product_category_name_english",
    columns="seller_state",
    aggfunc="sum",
    fill_value=0
)

# ---------------------------------------------------------------------
# 6️⃣ Contrôles
# ---------------------------------------------------------------------
assert pivot_category_state.shape[0] <= 6, (
    "❌ Plus de 6 catégories présentes dans le pivot."
)

assert pivot_category_state.shape[1] <= 5, (
    "❌ Plus de 5 États présents dans le pivot."
)

# ---------------------------------------------------------------------
# 7️⃣ Affichage
# ---------------------------------------------------------------------
print("\n🏆 TOP 6 CATÉGORIES × TOP 5 ÉTATS VENDEURS")
print("─" * 100)

print(
    pivot_category_state.to_string(
        float_format=lambda x: f"{x:,.2f}"
    )
)

print("\n" + "═" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
📊 CHIFFRE D'AFFAIRES — CATÉGORIES × ÉTATS VENDEURS
════════════════════════════════════════════════════════════════════════════════════════════════════

🏆 TOP 6 CATÉGORIES × TOP 5 ÉTATS VENDEURS
────────────────────────────────────────────────────────────────────────────────────────────────────
seller_state                          MG         PR         RJ        SC         SP
product_category_name_english                                                      
bed_bath_table                 27,947.03  15,506.38   6,338.26 57,122.27 909,463.04
computers_accessories         172,808.63 201,758.43  22,790.56 11,829.18 353,728.74
furniture_decor                57,990.02 134,750.94   5,027.13 10,490.59 498,629.41
health_beauty                  55,636.29 129,591.28 183,336.32 79,382.13 697,858.50
sports_leisure                 38,786.25 175,925.86  59,107.99 63,063.07 610,096.61
watches_gifts 

### Conclusion — Exercice 13

Le tableau croisé met en évidence une concentration du chiffre d'affaires sur certaines combinaisons **catégorie × État du vendeur**.

Par exemple, parmi les cinq États retenus, `watches_gifts` représente **971 086,60 BRL en SP**, `bed_bath_table` **909 463,04 BRL en SP** et `health_beauty` **697 858,50 BRL en SP**.

Le tableau montre également que l'État de **SP** concentre des montants importants pour plusieurs des six catégories analysées, tandis que d'autres catégories présentent des contributions plus importantes dans certains États comme `PR`, `RJ`, `MG` ou `SC`.

**Interprétation métier :** `pivot_table()` permet ici de passer d'une analyse séparée des catégories et des États à une lecture croisée de la performance commerciale. Cette matrice permet ainsi d'identifier les combinaisons **catégorie × territoire** qui concentrent le chiffre d'affaires et qui peuvent faire l'objet d'analyses complémentaires.

> **Périmètre :** le chiffre d'affaires correspond à la somme de `price` pour les 6 catégories et les 5 États sélectionnés. Le `freight_value` n'est pas inclus.


## Exercice 14 — Évolution mensuelle du chiffre d'affaires [8 pts]

Nous devons relier :

- `orders.order_purchase_timestamp`
- `items.price`

via `order_id`.

Puis calculer le chiffre d'affaires mensuel.

Questions :

1. Quel mois possède le CA le plus élevé ?
2. Le pic est-il environ 40 % supérieur aux mois voisins ?
3. Peut-on rapprocher ce pic d'un événement commercial connu ?
4. Pourquoi le premier et le dernier mois peuvent-ils être plus faibles ?

In [81]:

# ============================================================
# 📅 EXERCICE — ÉVOLUTION MENSUELLE DU CHIFFRE D'AFFAIRES
# ============================================================

print("\n" + "═" * 100)
print("📅 ÉVOLUTION MENSUELLE DU CHIFFRE D'AFFAIRES")
print("═" * 100)

# ---------------------------------------------------------------------
# 1️⃣ Préparation des dates de commande
# ---------------------------------------------------------------------
orders_dates = orders[
    [
        "order_id",
        "order_purchase_timestamp"
    ]
].copy()

# Conversion de la colonne en datetime
orders_dates["order_purchase_timestamp"] = pd.to_datetime(
    orders_dates["order_purchase_timestamp"],
    errors="coerce"
)

# ---------------------------------------------------------------------
# 2️⃣ Contrôle des dates
# ---------------------------------------------------------------------
dates_invalides = orders_dates["order_purchase_timestamp"].isna().sum()

print(
    f"📌 Dates de commande invalides ou manquantes : "
    f"{dates_invalides:,}"
)

assert dates_invalides == 0, (
    "❌ Certaines dates de commande sont invalides ou manquantes."
)

# ---------------------------------------------------------------------
# 3️⃣ Jointure entre les lignes de vente et les dates de commande
# ---------------------------------------------------------------------
sales_with_dates = items.merge(
    orders_dates,
    on="order_id",
    how="left",
    validate="many_to_one"
)

# La jointure ne doit pas modifier le nombre de lignes
assert len(sales_with_dates) == len(items), (
    "❌ La jointure a modifié le nombre de lignes."
)

# Toutes les lignes de vente doivent avoir une date de commande
assert (
    sales_with_dates["order_purchase_timestamp"]
    .notna()
    .all()
), (
    "❌ Certaines lignes de vente n'ont pas de date de commande."
)

print(
    f"📌 Lignes de vente analysées : "
    f"{len(sales_with_dates):,}"
)

# ---------------------------------------------------------------------
# 4️⃣ Création de la période mensuelle
# ---------------------------------------------------------------------
sales_with_dates["month"] = (
    sales_with_dates["order_purchase_timestamp"]
    .dt.to_period("M")
)

# ---------------------------------------------------------------------
# 5️⃣ Calcul du chiffre d'affaires mensuel
# ---------------------------------------------------------------------
monthly_revenue = (
    sales_with_dates
    .groupby("month")["price"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "price": "chiffre_affaires"
        }
    )
    .sort_values("month")
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------
# 6️⃣ Calcul de la variation mensuelle
# ---------------------------------------------------------------------
monthly_revenue["variation_pct"] = (
    monthly_revenue["chiffre_affaires"]
    .pct_change()
    * 100
)

# ---------------------------------------------------------------------
# 7️⃣ Affichage des résultats
# ---------------------------------------------------------------------
print("\n📈 CHIFFRE D'AFFAIRES PAR MOIS")
print("─" * 100)

print(
    monthly_revenue.to_string(
        index=False,
        formatters={
            "chiffre_affaires": "{:,.2f}".format,
            "variation_pct": (
                lambda x:
                "—" if pd.isna(x) else f"{x:.2f}%"
            )
        }
    )
)

print("\n" + "═" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
📅 ÉVOLUTION MENSUELLE DU CHIFFRE D'AFFAIRES
════════════════════════════════════════════════════════════════════════════════════════════════════
📌 Dates de commande invalides ou manquantes : 0
📌 Lignes de vente analysées : 112,650

📈 CHIFFRE D'AFFAIRES PAR MOIS
────────────────────────────────────────────────────────────────────────────────────────────────────
  month chiffre_affaires variation_pct
2016-09           267.36           NaN
2016-10        49,507.66     18417.23%
2016-12            10.90       -99.98%
2017-01       120,312.87   1103687.80%
2017-02       247,303.02       105.55%
2017-03       374,344.30        51.37%
2017-04       359,927.23        -3.85%
2017-05       506,071.14        40.60%
2017-06       433,038.60       -14.43%
2017-07       498,031.48        15.01%
2017-08       573,971.68        15.25%
2017-09       624,401.69         8.79%
2017-10       664,219.43    

In [83]:

# ============================================================
# 📈 EXERCICE — IDENTIFICATION DES PICS DE CA MENSUEL
# ============================================================

print("\n" + "═" * 100)
print("📈 ANALYSE DES PICs DE CHIFFRE D'AFFAIRES")
print("═" * 100)

# ---------------------------------------------------------------------
# 1️⃣ Vérification des données disponibles
# ---------------------------------------------------------------------
assert not monthly_revenue.empty, (
    "❌ monthly_revenue est vide."
)

assert "month" in monthly_revenue.columns, (
    "❌ La colonne 'month' est absente."
)

assert "chiffre_affaires" in monthly_revenue.columns, (
    "❌ La colonne 'chiffre_affaires' est absente."
)

# ---------------------------------------------------------------------
# 2️⃣ Identification du mois avec le CA maximal
# ---------------------------------------------------------------------
peak_index = (
    monthly_revenue["chiffre_affaires"]
    .idxmax()
)

peak_row = monthly_revenue.loc[peak_index]

print("\n🏆 MOIS AVEC LE CHIFFRE D'AFFAIRES MAXIMAL")
print("─" * 100)

print(
    f"📅 Mois de CA maximal : "
    f"{peak_row['month']}"
)

print(
    f"💰 CA maximal         : "
    f"{peak_row['chiffre_affaires']:,.2f} BRL"
)

# ---------------------------------------------------------------------
# 3️⃣ Identification des 5 meilleurs mois
# ---------------------------------------------------------------------
top_5_months = (
    monthly_revenue
    .sort_values(
        "chiffre_affaires",
        ascending=False
    )
    .head(5)
    .reset_index(drop=True)
)

print("\n🏅 TOP 5 DES MOIS PAR CHIFFRE D'AFFAIRES")
print("─" * 100)

print(
    top_5_months.to_string(
        index=False,
        formatters={
            "chiffre_affaires": "{:,.2f}".format,
            "variation_pct": (
                lambda x:
                "—" if pd.isna(x) else f"{x:.2f}%"
            )
        }
    )
)

# ---------------------------------------------------------------------
# 4️⃣ Premier mois observé
# ---------------------------------------------------------------------
premier_mois = (
    monthly_revenue
    .sort_values("month")
    .head(1)
)

print("\n📅 PREMIER MOIS OBSERVÉ")
print("─" * 100)

print(
    premier_mois.to_string(
        index=False,
        formatters={
            "chiffre_affaires": "{:,.2f}".format,
            "variation_pct": (
                lambda x:
                "—" if pd.isna(x) else f"{x:.2f}%"
            )
        }
    )
)

# ---------------------------------------------------------------------
# 5️⃣ Dernier mois observé
# ---------------------------------------------------------------------
dernier_mois = (
    monthly_revenue
    .sort_values("month")
    .tail(1)
)

print("\n📅 DERNIER MOIS OBSERVÉ")
print("─" * 100)

print(
    dernier_mois.to_string(
        index=False,
        formatters={
            "chiffre_affaires": "{:,.2f}".format,
            "variation_pct": (
                lambda x:
                "—" if pd.isna(x) else f"{x:.2f}%"
            )
        }
    )
)

print("\n" + "═" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
📈 ANALYSE DES PICs DE CHIFFRE D'AFFAIRES
════════════════════════════════════════════════════════════════════════════════════════════════════

🏆 MOIS AVEC LE CHIFFRE D'AFFAIRES MAXIMAL
────────────────────────────────────────────────────────────────────────────────────────────────────
📅 Mois de CA maximal : 2017-11
💰 CA maximal         : 1,010,271.37 BRL

🏅 TOP 5 DES MOIS PAR CHIFFRE D'AFFAIRES
────────────────────────────────────────────────────────────────────────────────────────────────────
  month chiffre_affaires variation_pct
2017-11     1,010,271.37        52.10%
2018-04       996,647.75         1.37%
2018-05       996,517.68        -0.01%
2018-03       983,213.44        16.47%
2018-01       950,030.36        27.71%

📅 PREMIER MOIS OBSERVÉ
────────────────────────────────────────────────────────────────────────────────────────────────────
  month chiffre_affaires variation_pct
2

In [84]:

# ============================================================
# 📅 EXERCICE — VÉRIFICATION DE LA COMPLÉTUDE DES MOIS
# ============================================================

print("\n" + "═" * 100)
print("📅 VÉRIFICATION DE LA COMPLÉTUDE DE LA PÉRIODE")
print("═" * 100)

# ---------------------------------------------------------------------
# 1️⃣ Identification de la première et de la dernière date d'achat
# ---------------------------------------------------------------------
min_date = (
    sales_with_dates["order_purchase_timestamp"]
    .min()
)

max_date = (
    sales_with_dates["order_purchase_timestamp"]
    .max()
)

print("\n📌 PÉRIODE OBSERVÉE")
print("─" * 100)

print(
    f"Première date d'achat : {min_date}"
)

print(
    f"Dernière date d'achat : {max_date}"
)

# ---------------------------------------------------------------------
# 2️⃣ Vérification du jour de début et de fin
# ---------------------------------------------------------------------
print("\n📅 POSITION DANS LE MOIS")
print("─" * 100)

print(
    f"Jour de la première date : {min_date.day}"
)

print(
    f"Jour de la dernière date : {max_date.day}"
)

print(
    f"Nombre de jours du dernier mois : "
    f"{max_date.days_in_month}"
)

# ---------------------------------------------------------------------
# 3️⃣ Interprétation du premier mois
# ---------------------------------------------------------------------
print("\n🔎 INTERPRÉTATION")
print("─" * 100)

if min_date.day > 1:
    print(
        "⚠️ Le premier mois est partiel : "
        "les observations commencent après le premier jour du mois."
    )
else:
    print(
        "✅ Le premier mois commence le premier jour du mois."
    )

# ---------------------------------------------------------------------
# 4️⃣ Interprétation du dernier mois
# ---------------------------------------------------------------------
if max_date.day < max_date.days_in_month:
    print(
        "⚠️ Le dernier mois est partiel : "
        "les observations s'arrêtent avant la fin du mois."
    )
else:
    print(
        "✅ Le dernier mois contient des observations "
        "jusqu'au dernier jour du mois."
    )

print("\n" + "═" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
📅 VÉRIFICATION DE LA COMPLÉTUDE DE LA PÉRIODE
════════════════════════════════════════════════════════════════════════════════════════════════════

📌 PÉRIODE OBSERVÉE
────────────────────────────────────────────────────────────────────────────────────────────────────
Première date d'achat : 2016-09-04 21:15:19
Dernière date d'achat : 2018-09-03 09:06:57

📅 POSITION DANS LE MOIS
────────────────────────────────────────────────────────────────────────────────────────────────────
Jour de la première date : 4
Jour de la dernière date : 3
Nombre de jours du dernier mois : 30

🔎 INTERPRÉTATION
────────────────────────────────────────────────────────────────────────────────────────────────────
⚠️ Le premier mois est partiel : les observations commencent après le premier jour du mois.
⚠️ Le dernier mois est partiel : les observations s'arrêtent avant la fin du mois.

══════════════════════════

### Conclusion — Exercice 14

Le chiffre d'affaires mensuel maximal est observé en **novembre 2017**, avec **1 010 271,37 BRL**, soit une progression de **52,10 %** par rapport à octobre 2017.

Ce pic peut être rapproché de la période du **Black Friday**, mais le dataset ne permet pas, à lui seul, d'établir une relation de causalité. Il s'agit donc d'une **hypothèse commerciale** qui devrait être confrontée à un calendrier externe ou à des données complémentaires.

Les bornes temporelles du dataset doivent également être prises en compte : la première commande date du **4 septembre 2016**, tandis que la dernière date du **3 septembre 2018**. Le premier mois et le dernier mois sont donc **partiels**.

Cela contribue à expliquer pourquoi leurs chiffres d'affaires sont très faibles (**267,36 BRL** en septembre 2016 et **145,00 BRL** en septembre 2018) par rapport aux mois situés au cœur de la période.

**Décision analytique :** les mois de bord ne doivent pas être comparés directement aux mois complets. Pour analyser l'évolution du niveau d'activité, il est préférable de privilégier les **mois complets**.


# PARTIE D — Transformer les données avec `apply` et `map`

## Exercice 15 — Classer les ventes par gamme de prix [8 pts]

Nous allons créer une nouvelle variable :

`gamme_prix`

avec trois catégories :

- Économique ;
- Standard ;
- Premium.

### Seuils retenus

Pour cet exercice, nous choisissons :

- Économique : prix < 50 BRL ;
- Standard : 50 ≤ prix < 200 BRL ;
- Premium : prix ≥ 200 BRL.

Ces seuils sont une convention analytique. Ils pourraient être modifiés selon
le contexte commercial.

In [85]:

# ============================================================
# 💰 EXERCICE — SEGMENTATION DES VENTES PAR GAMME DE PRIX
# ============================================================

print("\n" + "═" * 100)
print("💰 SEGMENTATION DES VENTES PAR GAMME DE PRIX")
print("═" * 100)


# ---------------------------------------------------------------------
# 1️⃣ Fonction de classification des prix
# ---------------------------------------------------------------------
def classer_gamme_prix(price):
    """
    Classe un prix dans une gamme commerciale.

    Règles :
    - Économique : prix < 50 BRL
    - Standard   : 50 <= prix < 200 BRL
    - Premium    : prix >= 200 BRL
    """

    if price < 50:
        return "Économique"

    elif price < 200:
        return "Standard"

    else:
        return "Premium"


# ---------------------------------------------------------------------
# 2️⃣ Application de la classification
# ---------------------------------------------------------------------
df["gamme_prix"] = (
    df["price"]
    .apply(classer_gamme_prix)
)


# ---------------------------------------------------------------------
# 3️⃣ Vérification des valeurs obtenues
# ---------------------------------------------------------------------
gammes_attendues = {
    "Économique",
    "Standard",
    "Premium"
}

gammes_observees = set(
    df["gamme_prix"].dropna().unique()
)

assert gammes_observees.issubset(gammes_attendues), (
    "❌ Une gamme de prix inattendue a été détectée."
)


# ---------------------------------------------------------------------
# 4️⃣ Calcul du nombre de ventes par gamme
# ---------------------------------------------------------------------
gamme_summary = (
    df["gamme_prix"]
    .value_counts()
    .rename_axis("gamme_prix")
    .reset_index(name="nombre_ventes")
)


# ---------------------------------------------------------------------
# 5️⃣ Calcul des proportions
# ---------------------------------------------------------------------
gamme_summary["proportion_pct"] = (
    gamme_summary["nombre_ventes"]
    / gamme_summary["nombre_ventes"].sum()
    * 100
)


# ---------------------------------------------------------------------
# 6️⃣ Contrôles de cohérence
# ---------------------------------------------------------------------
assert (
    gamme_summary["nombre_ventes"].sum()
    == len(df)
), "❌ Le total des ventes ne correspond pas au nombre de lignes de df."

assert (
    round(gamme_summary["proportion_pct"].sum(), 10)
    == 100
), "❌ Les proportions ne totalisent pas 100 %."


# ---------------------------------------------------------------------
# 7️⃣ Affichage des résultats
# ---------------------------------------------------------------------
print("\n📊 RÉPARTITION DES VENTES PAR GAMME")
print("─" * 100)

print(
    gamme_summary.to_string(
        index=False,
        formatters={
            "proportion_pct": "{:.2f}%".format
        }
    )
)

print("\n" + "═" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
💰 SEGMENTATION DES VENTES PAR GAMME DE PRIX
════════════════════════════════════════════════════════════════════════════════════════════════════

📊 RÉPARTITION DES VENTES PAR GAMME
────────────────────────────────────────────────────────────────────────────────────────────────────
gamme_prix  nombre_ventes proportion_pct
  Standard          60153         53.40%
Économique          39024         34.64%
   Premium          13473         11.96%

════════════════════════════════════════════════════════════════════════════════════════════════════


### Conclusion — Exercice 15

Avec les seuils retenus pour cette classification :

* **Standard** : 60 153 lignes de vente, soit **53,40 %** ;
* **Économique** : 39 024 lignes de vente, soit **34,64 %** ;
* **Premium** : 13 473 lignes de vente, soit **11,96 %**.

La gamme **Standard** représente donc la majorité des lignes de vente analysées selon cette classification, suivie de la gamme **Économique**. La gamme **Premium** représente une proportion plus faible des lignes observées.

Ces seuils constituent une **règle analytique définie pour l'exercice** et ne doivent pas être présentés comme une segmentation commerciale officielle d'Olist.

**Conclusion :** l'utilisation de `apply()` permet de transformer la variable quantitative `price` en une variable catégorielle `gamme_prix`, facilitant ainsi l'analyse et la comparaison des différents segments de prix.


## Exercice 16 — `map()` : vendeur → État [6 pts]

Nous voulons uniquement récupérer l'État correspondant à chaque vendeur.

Relation :

`seller_id → seller_state`

Pour cette correspondance simple, `map()` est une alternative légère à un
`merge()` complet.

In [86]:

# ============================================================
# 🗺️ EXERCICE — ASSOCIATION VENDEUR → ÉTAT AVEC map()
# ============================================================

print("\n" + "═" * 100)
print("🗺️  ENRICHISSEMENT DES VENTES AVEC L'ÉTAT DU VENDEUR")
print("═" * 100)


# ---------------------------------------------------------------------
# 1️⃣ Création de la table de correspondance
# ---------------------------------------------------------------------
# On construit une Series où :
#     index  = seller_id
#     valeur = seller_state
#
# Elle servira de dictionnaire de correspondance pour map().
seller_state_map = (
    sellers
    .set_index("seller_id")["seller_state"]
)

print(
    f"\n📌 Nombre de vendeurs dans la table de correspondance : "
    f"{len(seller_state_map):,}"
)


# ---------------------------------------------------------------------
# 2️⃣ Création d'une copie des lignes de vente
# ---------------------------------------------------------------------
items_with_state = items.copy()


# ---------------------------------------------------------------------
# 3️⃣ Ajout de l'État du vendeur avec map()
# ---------------------------------------------------------------------
items_with_state["seller_state"] = (
    items_with_state["seller_id"]
    .map(seller_state_map)
)


# ---------------------------------------------------------------------
# 4️⃣ Vérification des valeurs manquantes
# ---------------------------------------------------------------------
missing_states = (
    items_with_state["seller_state"]
    .isna()
    .sum()
)

print(
    f"📌 États vendeurs manquants après map() : "
    f"{missing_states:,}"
)

assert missing_states == 0, (
    "❌ Certains seller_id n'ont pas trouvé de seller_state."
)


# ---------------------------------------------------------------------
# 5️⃣ Vérification du nombre de lignes
# ---------------------------------------------------------------------
assert len(items_with_state) == len(items), (
    "❌ Le nombre de lignes a changé."
)


# ---------------------------------------------------------------------
# 6️⃣ Aperçu du résultat
# ---------------------------------------------------------------------
print("\n🔎 APERÇU DE LA CORRESPONDANCE")
print("─" * 100)

print(
    items_with_state[
        [
            "seller_id",
            "seller_state"
        ]
    ]
    .head()
    .to_string(index=False)
)

print("\n" + "═" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
🗺️  ENRICHISSEMENT DES VENTES AVEC L'ÉTAT DU VENDEUR
════════════════════════════════════════════════════════════════════════════════════════════════════

📌 Nombre de vendeurs dans la table de correspondance : 3,095
📌 États vendeurs manquants après map() : 0

🔎 APERÇU DE LA CORRESPONDANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
                       seller_id seller_state
48436dade18ac8b2bce089ec2a041202           SP
dd7ddc04e1b6c2c614352b383efe2d36           SP
5b51032eddd242adc84c38acab88f23d           MG
9d7a1d34a5052409006425275ba1c2b4           SP
df560393f3a51e74553ab94004ba5c87           PR

════════════════════════════════════════════════════════════════════════════════════════════════════


### Conclusion — Exercice 16

Le mapping `seller_id → seller_state` a permis d’enrichir la table des ventes avec l’État du vendeur.

Le contrôle réalisé montre **0 État vendeur manquant** après l’utilisation de `map()`, ce qui confirme que chaque `seller_id` présent dans `items` possède une correspondance dans la table `sellers`.

Cette méthode est particulièrement adaptée lorsqu’on souhaite effectuer une **correspondance simple clé → valeur**, sans avoir besoin de joindre plusieurs colonnes de la table de référence.

De plus, `map()` permet ici de conserver le **grain initial des ventes**, puisqu’il ajoute une information à chaque ligne sans créer de nouvelles lignes.

**Conclusion :** `map()` constitue donc une méthode simple et efficace pour enrichir une table avec une seule information provenant d'une table de référence, comme ici `seller_id → seller_state`. La variable `seller_state` peut ensuite être utilisée pour réaliser des analyses géographiques des ventes.


### EXERCICE 17 — Synthèse pour le comité commercial [6 pts]

La synthèse finale doit contenir **6 à 10 phrases**.

Elle doit utiliser au minimum **5 chiffres réellement calculés**.

La synthèse doit répondre à :

1. quelles catégories semblent importantes ;
2. quels États concentrent le CA ;
3. comment les ventes se répartissent dans le temps ;
4. quelle proportion des ventes est locale ;
5. quel est le niveau de satisfaction ;
6. quelle action commerciale proposer pour le prochain trimestre.

### Attention

Il ne faut pas inventer les valeurs.

Les chiffres doivent être repris des tableaux calculés précédemment.

In [93]:

# ============================================================
# ⭐ EXERCICE — AGRÉGATION DES AVIS PAR COMMANDE
# ============================================================

print("\n" + "═" * 100)
print("⭐ AGRÉGATION DES AVIS PAR COMMANDE")
print("═" * 100)


# ---------------------------------------------------------------------
# 0️⃣ NETTOYAGE DES ANCIENNES COLONNES D'AVIS
# ---------------------------------------------------------------------
# Cette étape rend la cellule réexécutable.
# Elle évite les conflits avec d'anciens review_score_x / review_score_y
# ou review_count_x / review_count_y créés lors d'une exécution précédente.

colonnes_reviews = [
    "review_score",
    "review_count",
    "review_score_x",
    "review_score_y",
    "review_count_x",
    "review_count_y"
]

colonnes_presentes = [
    colonne
    for colonne in colonnes_reviews
    if colonne in df.columns
]

if colonnes_presentes:
    print(
        "\n🧹 Anciennes colonnes d'avis supprimées :",
        ", ".join(colonnes_presentes)
    )

    df = df.drop(columns=colonnes_presentes)


# ---------------------------------------------------------------------
# 1️⃣ CRÉATION DE LA TABLE DE SYNTHÈSE DES AVIS
# ---------------------------------------------------------------------

reviews_summary = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_score", "count")
    )
)

print(
    f"\n📌 Nombre de commandes avec avis : "
    f"{reviews_summary['order_id'].nunique():,}"
)

print(
    f"📌 Nombre de lignes dans reviews_summary : "
    f"{len(reviews_summary):,}"
)


# ---------------------------------------------------------------------
# 2️⃣ VÉRIFICATION DE L'UNICITÉ DE order_id
# ---------------------------------------------------------------------

assert reviews_summary["order_id"].is_unique, (
    "❌ reviews_summary contient plusieurs lignes par order_id."
)

print(
    "📌 Unicité de order_id : OK"
)


# ---------------------------------------------------------------------
# 3️⃣ AJOUT DES AVIS AU DATAFRAME PRINCIPAL
# ---------------------------------------------------------------------

lignes_avant = len(df)

df = df.merge(
    reviews_summary,
    on="order_id",
    how="left",
    validate="many_to_one"
)

lignes_apres = len(df)


# ---------------------------------------------------------------------
# 4️⃣ CONTRÔLE DE CONSERVATION DU GRAIN
# ---------------------------------------------------------------------

print(f"\n📌 Lignes avant le merge : {lignes_avant:,}")
print(f"📌 Lignes après le merge  : {lignes_apres:,}")
print(f"📌 Variation              : {lignes_apres - lignes_avant:+,}")

assert lignes_avant == lignes_apres, (
    "❌ Le nombre de lignes a changé après le merge."
)

print("✅ Grain du DataFrame préservé.")


# ---------------------------------------------------------------------
# 5️⃣ CONTRÔLE DES VALEURS MANQUANTES
# ---------------------------------------------------------------------

avis_manquants = df["review_score"].isna().sum()

print(
    f"\n📌 Lignes sans review_score : "
    f"{avis_manquants:,}"
)


# ---------------------------------------------------------------------
# 6️⃣ APERÇU DES DONNÉES ENRICHIES
# ---------------------------------------------------------------------

print("\n🔎 APERÇU DES DONNÉES ENRICHIES")
print("─" * 100)

print(
    df[
        [
            "order_id",
            "review_score",
            "review_count"
        ]
    ]
    .head()
    .to_string(index=False)
)


# ---------------------------------------------------------------------
# 7️⃣ CONTRÔLE FINAL
# ---------------------------------------------------------------------

assert "review_score" in df.columns, (
    "❌ review_score n'a pas été ajoutée à df."
)

assert "review_count" in df.columns, (
    "❌ review_count n'a pas été ajoutée à df."
)

print("\n✅ review_score et review_count sont maintenant disponibles dans df.")

print("\n" + "═" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
⭐ AGRÉGATION DES AVIS PAR COMMANDE
════════════════════════════════════════════════════════════════════════════════════════════════════

🧹 Anciennes colonnes d'avis supprimées : review_score, review_count, review_score_x, review_score_y, review_count_x, review_count_y

📌 Nombre de commandes avec avis : 98,673
📌 Nombre de lignes dans reviews_summary : 98,673
📌 Unicité de order_id : OK

📌 Lignes avant le merge : 112,650
📌 Lignes après le merge  : 112,650
📌 Variation              : +0
✅ Grain du DataFrame préservé.

📌 Lignes sans review_score : 942

🔎 APERÇU DES DONNÉES ENRICHIES
────────────────────────────────────────────────────────────────────────────────────────────────────
                        order_id  review_score  review_count
00010242fe8c5a6d1ba2dd792cb16214           5.0           1.0
00018f77f2f0320c557190d7a144bdd3           4.0           1.0
000229ec398224ef6ca0657da4fc70

In [94]:

# ============================================================
# 📊 KPI — SYNTHÈSE GÉNÉRALE DE L'ANALYSE
# ============================================================

print("\n" + "═" * 100)
print("📊 KPI — SYNTHÈSE GÉNÉRALE")
print("═" * 100)


# ---------------------------------------------------------------------
# 🔎 0️⃣ VÉRIFICATION DES COLONNES NÉCESSAIRES
# ---------------------------------------------------------------------

colonnes_requises = [
    "price",
    "order_id",
    "seller_id",
    "product_category_name_english",
    "vente_locale"
]

colonnes_manquantes = [
    colonne
    for colonne in colonnes_requises
    if colonne not in df.columns
]

if colonnes_manquantes:
    raise KeyError(
        "❌ Colonnes manquantes dans df : "
        + ", ".join(colonnes_manquantes)
    )

if "review_score" not in reviews_summary.columns:
    raise KeyError(
        "❌ La colonne 'review_score' est absente de reviews_summary."
    )


# ---------------------------------------------------------------------
# 1️⃣ CHIFFRE D'AFFAIRES TOTAL
# ---------------------------------------------------------------------

ca_total = df["price"].sum()


# ---------------------------------------------------------------------
# 2️⃣ NOMBRE DE LIGNES DE VENTE
# ---------------------------------------------------------------------

nombre_lignes_vente = len(df)


# ---------------------------------------------------------------------
# 3️⃣ NOMBRE DE COMMANDES DISTINCTES
# ---------------------------------------------------------------------

nombre_commandes = df["order_id"].nunique()


# ---------------------------------------------------------------------
# 4️⃣ NOMBRE DE VENDEURS DISTINCTS
# ---------------------------------------------------------------------

nombre_vendeurs = df["seller_id"].nunique()


# ---------------------------------------------------------------------
# 5️⃣ NOMBRE DE CATÉGORIES DISTINCTES
# ---------------------------------------------------------------------

nombre_categories = (
    df["product_category_name_english"]
    .nunique()
)


# ---------------------------------------------------------------------
# 6️⃣ PRIX MOYEN PAR LIGNE DE VENTE
# ---------------------------------------------------------------------

prix_moyen = df["price"].mean()


# ---------------------------------------------------------------------
# 7️⃣ PROPORTION DES VENTES LOCALES
# ---------------------------------------------------------------------

proportion_locale = (
    df["vente_locale"].mean() * 100
)


# ---------------------------------------------------------------------
# 8️⃣ SCORE MOYEN DES AVIS
# ---------------------------------------------------------------------
# Le calcul est effectué sur reviews_summary afin que
# chaque commande contribue une seule fois au KPI.

score_moyen = (
    reviews_summary["review_score"]
    .mean()
)


# ---------------------------------------------------------------------
# 9️⃣ CONSTRUCTION DU TABLEAU DE SYNTHÈSE
# ---------------------------------------------------------------------

kpi_summary = pd.DataFrame({
    "KPI": [
        "CA total produits",
        "Nombre de lignes de vente",
        "Nombre de commandes",
        "Nombre de vendeurs",
        "Nombre de catégories",
        "Prix moyen",
        "Proportion ventes locales",
        "Score moyen"
    ],
    "Valeur": [
        ca_total,
        nombre_lignes_vente,
        nombre_commandes,
        nombre_vendeurs,
        nombre_categories,
        prix_moyen,
        proportion_locale,
        score_moyen
    ],
    "Unité": [
        "BRL",
        "lignes",
        "commandes",
        "vendeurs",
        "catégories",
        "BRL",
        "%",
        "/5"
    ]
})


# ---------------------------------------------------------------------
# 🔎 AFFICHAGE DE LA SYNTHÈSE
# ---------------------------------------------------------------------

print("\n📋 TABLEAU DES KPI")
print("─" * 100)

print(
    kpi_summary.to_string(index=False)
)

print("\n" + "═" * 100)




════════════════════════════════════════════════════════════════════════════════════════════════════
📊 KPI — SYNTHÈSE GÉNÉRALE
════════════════════════════════════════════════════════════════════════════════════════════════════

📋 TABLEAU DES KPI
────────────────────────────────────────────────────────────────────────────────────────────────────
                      KPI       Valeur      Unité
        CA total produits 1.359164e+07        BRL
Nombre de lignes de vente 1.126500e+05     lignes
      Nombre de commandes 9.866600e+04  commandes
       Nombre de vendeurs 3.095000e+03   vendeurs
     Nombre de catégories 7.100000e+01 catégories
               Prix moyen 1.206537e+02        BRL
Proportion ventes locales 3.617932e+01          %
              Score moyen 4.086793e+00         /5

════════════════════════════════════════════════════════════════════════════════════════════════════


## Synthèse finale — Analyse Olist

L'analyse porte sur **112 650 lignes de vente**, **98 666 commandes distinctes** et un chiffre d'affaires total des produits de **13 591 643,70 BRL**. Le prix moyen par ligne de vente s'établit à **120,65 BRL**.

Du point de vue des catégories, les trois catégories générant le plus de chiffre d'affaires sont `health_beauty` (**1 258 681,34 BRL**), `watches_gifts` (**1 205 005,68 BRL**) et `bed_bath_table` (**1 036 988,68 BRL**).

Sur le plan géographique, l'État de **SP** présente le chiffre d'affaires le plus élevé avec **8 753 396,21 BRL**, suivi de **PR** avec **1 261 887,21 BRL** et de **MG** avec **1 011 564,74 BRL**. Cette répartition montre une distribution géographique inégale du chiffre d'affaires entre les États vendeurs.

Les ventes dites **nationales représentent 63,82 % des lignes de vente**, contre **36,18 % pour les ventes locales**. Dans cette analyse, une vente est considérée comme locale lorsque `seller_state == customer_state`. Cette définition permet d'étudier la proximité au niveau des États, mais ne mesure pas directement la distance géographique réelle.

L'analyse temporelle montre un chiffre d'affaires mensuel maximal en **novembre 2017**, avec **1 010 271,37 BRL**, soit une progression de **52,10 % par rapport à octobre 2017**. Cette période correspond à celle du Black Friday, ce qui constitue une piste d'interprétation, mais les données analysées ne permettent pas à elles seules d'établir une relation causale.

Concernant la satisfaction, le **score moyen des avis est de 4,09/5**, calculé au niveau commande à partir de `reviews_summary`. Par ailleurs, les lignes de vente sont majoritairement classées dans la gamme **Standard (53,40 %)**, selon les seuils définis dans l'analyse : moins de 50 BRL pour « Économique », de 50 à moins de 200 BRL pour « Standard » et 200 BRL ou plus pour « Premium ».

Enfin, l'analyse met en évidence plusieurs dimensions complémentaires à suivre : **valeur des catégories, performance géographique des vendeurs, satisfaction client, volume des ventes et prix moyen**. Leur analyse conjointe permettrait d'approfondir l'identification des principaux moteurs du chiffre d'affaires et des différences de performance entre vendeurs, catégories et territoires.

> **Note méthodologique :** les indicateurs de cette synthèse reposent principalement sur le périmètre de **112 650 lignes de vente** du DataFrame final. Le nombre de **98 673 commandes dans `reviews_summary`** correspond spécifiquement aux commandes présentes dans la table agrégée des avis et ne doit pas être confondu avec les **98 666 commandes distinctes du périmètre de vente analysé**.


# CONTRÔLES FINAUX DU LIVRABLE

Avant de remettre le notebook, vérifier :

- [ ] les 7 tables sont chargées ;
- [ ] l'EDA est réalisée avant les premiers `merge` ;
- [ ] les valeurs manquantes sont analysées ;
- [ ] `review_id` est contrôlé ;
- [ ] les exercices 4 à 8 utilisent `merge()` ;
- [ ] l'Exercice 7 montre l'effet de la cardinalité ;
- [ ] les reviews sont agrégées avant les KPI commerciaux ;
- [ ] les exercices 9 à 12 utilisent `groupby()` ;
- [ ] l'Exercice 13 utilise `pivot_table()` ;
- [ ] l'Exercice 14 calcule le CA mensuel ;
- [ ] l'Exercice 15 utilise `apply()` ;
- [ ] l'Exercice 16 utilise `map()` ;
- [ ] l'Exercice 17 utilise au moins 5 chiffres calculés ;
- [ ] aucune visualisation n'est ajoutée ;
- [ ] le notebook est exécuté de haut en bas avec `Run All`.

# Conclusion générale

Ce cas pratique met en évidence une démarche complète de Data Analyst : **auditer les données avant de les fusionner, contrôler les cardinalités, préserver le grain des données, documenter les anomalies, construire des indicateurs fiables puis interpréter les résultats dans leur contexte métier**.

Le traitement réalisé a permis de conserver les anomalies identifiées lorsqu'elles contenaient une information utile, de les documenter ou de les matérialiser par des indicateurs plutôt que de les supprimer sans justification. Le DataFrame analytique final conserve **112 650 lignes de vente**, pour un chiffre d'affaires produit de **13 591 643,70 BRL** et un fret total de **2 251 909,54 BRL**.

### Réponse au problème métier

L'analyse montre plusieurs leviers importants à considérer pour comprendre la performance commerciale et logistique d'Olist.

Sur le plan **géographique**, le chiffre d'affaires est particulièrement élevé chez les vendeurs de **São Paulo (SP)** avec **8 753 396,21 BRL**, devant **Paraná (PR)** avec **1 261 887,21 BRL** et **Minas Gerais (MG)** avec **1 011 564,74 BRL**. L'analyse des ventes locales et nationales montre également que **63,82 % des lignes de vente sont nationales**, contre **36,18 % de ventes locales**, selon la définition retenue de la vente locale (`seller_state == customer_state`). Ces résultats indiquent que la dimension géographique doit être étudiée conjointement avec les flux logistiques et la performance des vendeurs.

Sur le plan **catalogue**, les catégories `health_beauty`, `watches_gifts` et `bed_bath_table` présentent les chiffres d'affaires les plus élevés, avec respectivement **1 258 681,34 BRL**, **1 205 005,68 BRL** et **1 036 988,68 BRL**. La segmentation par gamme de prix montre par ailleurs que les lignes de vente sont majoritairement classées dans la gamme **Standard (53,40 %)**, suivie de la gamme **Économique (34,64 %)** et de la gamme **Premium (11,96 %)**, selon les seuils définis dans l'étude.

Du côté de la **satisfaction**, le score moyen des avis est de **4,09/5** au niveau commande. Cet indicateur doit être analysé avec les dimensions géographiques, catégorielles et logistiques afin de déterminer si certaines zones, catégories ou gammes présentent simultanément un niveau important de valeur commerciale et des différences de satisfaction.

### Implication pour les investissements

Les résultats permettent donc de **cibler les axes d'investigation et de priorisation**, mais ils ne suffisent pas à eux seuls à déterminer un investissement optimal. Une étape complémentaire consisterait à croiser systématiquement **chiffre d'affaires, volume, gamme de prix, satisfaction et indicateurs logistiques** par État et par catégorie.

L'objectif serait notamment d'identifier :

* les **États** qui concentrent le plus de valeur commerciale ;
* les **catégories** qui génèrent le plus de chiffre d'affaires ;
* les **gammes de prix** qui représentent le plus grand volume de ventes ;
* les couples **État × catégorie** qui concentrent la valeur ;
* les zones ou catégories présentant un écart entre **performance commerciale et satisfaction client** ;
* les situations où le **fret et la dimension logistique** peuvent constituer un facteur important à approfondir.

Ainsi, l'analyse ne fournit pas simplement un tableau de chiffres : elle construit une **base factuelle pour orienter les prochaines analyses commerciales et logistiques du trimestre**. Les décisions d'investissement devront ensuite être prises à partir de ces indicateurs croisés, complétés par des analyses de coûts, de marges, de délais de livraison et de satisfaction.

**Le principal enseignement du cas est donc double : la qualité de la décision dépend d'abord de la qualité du traitement des données, puis de la capacité à transformer les indicateurs obtenus en questions métier pertinentes.**
